# Deep Multi-Task (MLP-based) — Reconstruction + Anomaly + 7-Class Classifier

One model, 3 outputs: reconstruction (30x7), anomaly sigmoid, 7-class softmax.

Reconstruction is trained on the BROADER safe set (normal + motor_off + transition_on + transition_off) — motor_off/transitions are legitimate states, not anomalies, so the decoder needs to have seen them too, or every real shutdown/startup would falsely trigger a high reconstruction error. Kept small (64->32->16 encoder) since this is the model most likely to actually ship to the MCU.

Complete three-head multi-task workflow using only Dense layers inside the neural network:
1. Dense autoencoder reconstruction head
2. Dense binary safe/fault head
3. Dense seven-class state/fault classifier

The data loading, session-wise cross-validation, safe-only reconstruction training,
state/PWM/load-conditioned thresholds, decision fusion, exports, and manual testing
remain the same as in the Conv1D version.

In [ ]:
# ==========================================================
# CELL 0 — install dependencies
# ==========================================================
%pip install pandas numpy scikit-learn joblib openpyxl matplotlib tensorflow tf2onnx onnx

In [ ]:
# ==========================================================
# CELL 1 — imports
# ==========================================================
import os
import glob
import json
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import (
    StratifiedGroupKFold,
    StratifiedShuffleSplit,
)
from sklearn.utils.class_weight import compute_class_weight

In [ ]:
# ==========================================================
# CELL 2 — configuration
# ==========================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
tf.config.experimental.enable_op_determinism()
BASE_PATH = (
    r"../../data/Motor_data_model"
)
SCALE_DIVISOR = 1_000_000.0
FEATURE_COLUMNS = [
    "T_slope_10s",
    "PWM",
    "Irms",
    "Vrms",
    "Current_Variance",
    "Current_Crest_Factor",
    "Vib_RMS",
    "Vib_Min",
    "Vib_Q1",
    "Vib_Median",
    "Vib_Q3",
    "Vib_Max",
    "Vib_Variance",
    "Vib_Skewness",
]
FINAL_LABELS = [
    "motor_off",
    "turning_off",
    "turning_on",
    "normal",
    "electrical_fault",
    "mechanical_fault",
    "thermal_fault",
]
LABEL_TO_ID = {
    label: index
    for index, label in enumerate(FINAL_LABELS)
}
SAFE_LABELS = [
    "motor_off",
    "turning_off",
    "turning_on",
    "normal",
]
FAULT_LABELS = [
    "electrical_fault",
    "mechanical_fault",
    "thermal_fault",
]
SAFE_CLASS_IDS = np.asarray(
    [LABEL_TO_ID[label] for label in SAFE_LABELS],
    dtype=np.int64,
)
FAULT_CLASS_IDS = np.asarray(
    [LABEL_TO_ID[label] for label in FAULT_LABELS],
    dtype=np.int64,
)
N_FEATURES = len(FEATURE_COLUMNS)
N_CLASSES = len(FINAL_LABELS)
WINDOW_LEN = 30
WINDOW_STEP = 10
ROW_PERIOD_SEC = 0.015
N_SPLITS = 5
BATCH_SIZE = 128
MAX_EPOCHS = 180
LEARNING_RATE = 1e-3
RECON_LOSS_WEIGHT = 1.00
ANOMALY_LOSS_WEIGHT = 0.80
CLASS_LOSS_WEIGHT = 1.00
# Compact pure-MLP architecture.
ENCODER_DENSE_1 = 32        #32
ENCODER_DENSE_2 = 24           #24
DROPOUT_LAYER_1 = 0.10
DROPOUT_LAYER_2 = 0.05
LATENT_DENSE_UNITS1 = 8     #8
LATENT_DENSE_UNITS2 = 10   #10
DECODER_DENSE_1 = 24        #24
DECODER_DENSE_2 = 32        #32
CLASS_DENSE_UNITS = 24
ANOMALY_DENSE_UNITS = 12
CLASS_CONF_THRESHOLD = 0.65
ANOMALY_PROB_THRESHOLD = 0.65
RECON_QUANTILE = 0.990
# --- reconstruction severity tiers (notebook-only, no new data) ---
SEVERITY_MILD_MULTIPLIER = 2.0
SEVERITY_MODERATE_MULTIPLIER = 4.0
SEVERITY_SEVERE_MULTIPLIER = 8.0
# ==========================================================
# CHANGED: PWM bin edges.
# 0-30% is now ONE merged bin instead of three 10%-wide bins.
# Widening/smoothing could not stabilize that range because it
# isn't a sample-size problem: your low-PWM "normal-with-load"
# files are ramp recordings, so low-PWM windows are genuinely
# mixed with acceleration-transient content -- real signal, not
# noise. One wide bin with its own correctly-higher threshold is
# the right model for that region. 30-100% keeps 10%-wide bins,
# already proven stable (3.5-7.7x threshold/baseline ratio).
# ==========================================================
PWM_BIN_EDGES = np.asarray(
    [0.0, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90, 1.00],
    dtype=np.float32,
)
 
PWM_BIN_NAMES = [
    f"{int(round(PWM_BIN_EDGES[i]*100))}_{int(round(PWM_BIN_EDGES[i+1]*100))}"
    for i in range(len(PWM_BIN_EDGES) - 1)
]
LOAD_PROXY_FEATURE = "Irms"
MIN_THRESHOLD_WINDOWS = 30
OUTPUT_DIR = "mlp_ae_ann_3head_output"
CV_DIR = os.path.join(
    OUTPUT_DIR,
    "cross_validation",
)
FINAL_DIR = os.path.join(
    OUTPUT_DIR,
    "final_model",
)
MANUAL_TEST_DIR = os.path.join(
    OUTPUT_DIR,
    "manual_test_results",
)
for folder in [
    CV_DIR,
    FINAL_DIR,
    MANUAL_TEST_DIR,
]:
    os.makedirs(
        folder,
        exist_ok=True,
    )
print("Model: Pure MLP ANN 3-Head Hybrid")
print("Input:", WINDOW_LEN, "x", N_FEATURES)
print("PWM bins:", len(PWM_BIN_NAMES), PWM_BIN_NAMES)
print("Classes:", FINAL_LABELS)
print("Safe classes:", SAFE_LABELS)
print("Fault classes:", FAULT_LABELS)
 
 

In [ ]:
import glob, os
import pandas as pd

files = glob.glob(os.path.join(BASE_PATH, "**", "*.xlsx"), recursive=True) + \
        glob.glob(os.path.join(BASE_PATH, "**", "*.csv"), recursive=True)

scale_report = []
for path in files:
    try:
        df = pd.read_excel(path) if path.endswith(".xlsx") else pd.read_csv(path)
        if "PWM" not in df.columns:
            continue
        pwm_median = pd.to_numeric(df["PWM"], errors="coerce").median()
        scale_report.append((os.path.relpath(path, BASE_PATH), pwm_median))
    except Exception as e:
        scale_report.append((os.path.relpath(path, BASE_PATH), f"ERROR: {e}"))

report_df = pd.DataFrame(scale_report, columns=["file", "pwm_median_raw"])
report_df.to_csv("scale_check_report.csv", index=False)
print(report_df.sort_values("pwm_median_raw").to_string())

In [ ]:
# ==========================================================
# CELL 3 — deterministic file-to-class mapping
# ==========================================================
def normalized_path(path):
    return str(path).replace("\\", "/").lower()


def label_for(path):
    path_text = normalized_path(path)
    stem = Path(path).stem.lower()

    if any(term in path_text for term in [
        "motor_being_off",
        "turning_off",
        "transition_off",
        "off_spikes",
    ]):
        return "turning_off"

    if any(term in path_text for term in [
        "motor_being_on",
        "turning_on",
        "transition_on",
        "on_spikes",
    ]):
        return "turning_on"

    if "/motor_off/" in path_text or "motor_off" in stem:
        return "motor_off"

    if any(term in path_text for term in [
        "electrical fault",
        "electrical_fault",
        "abnormal_current",
    ]):
        return "electrical_fault"

    if any(term in path_text for term in [
        "mechanical fault",
        "mechanical_fault",
        "abnormal_vibration",
    ]):
        return "mechanical_fault"

    if any(term in path_text for term in [
        "thermal fault",
        "thermal_fault",
        "abnormal_temp",
    ]):
        return "thermal_fault"

    if (
        "/normal/" in path_text
        or stem.startswith("normal_")
        or stem == "normal"
    ):
        return "normal"

    return None


def anomaly_target_from_class_id(class_id):
    return int(
        class_id in FAULT_CLASS_IDS
    )

def is_normal_with_load_file(path, label):
    if label != "normal":
        return False

    path_text = normalized_path(path)

    # Check without-load first because it also contains the word "load".
    without_load_terms = [
        "without_load",
        "without load",
        "no_load",
        "no load",
        "unloaded",
        "normal_noload",
        "normal_no_load",
    ]

    with_load_terms = [
        "with_load",
        "with load",
        "loaded",
        "normal_load",
    ]

    if any(term in path_text for term in without_load_terms):
        return False

    if any(term in path_text for term in with_load_terms):
        return True

    raise ValueError(
        "Cannot identify loaded/unloaded normal file: "
        f"{path}"
    )

In [ ]:
# ==========================================================
# CELL 4 — discover and read files
# ==========================================================
def discover_files(base_path):
    files = glob.glob(
        os.path.join(base_path, "**", "*.xlsx"),
        recursive=True,
    )

    files += glob.glob(
        os.path.join(base_path, "**", "*.csv"),
        recursive=True,
    )

    return sorted(set(files))


def read_one(path):
    path = Path(path)

    if path.suffix.lower() == ".xlsx":
        df = pd.read_excel(path)
    elif path.suffix.lower() == ".csv":
        df = pd.read_csv(path)
    else:
        raise ValueError(f"Unsupported file type: {path}")

    missing_columns = [c for c in FEATURE_COLUMNS if c not in df.columns]
    if missing_columns:
        raise ValueError(f"Missing columns in {path.name}: {missing_columns}")

    clean = df[FEATURE_COLUMNS].apply(pd.to_numeric, errors="coerce")
    clean = clean / SCALE_DIVISOR

    # ADDED: PWM duty cycle is physically bounded to [0, 1]. Values
    # recorded slightly above 1.0 are integer-logging artifacts at
    # the 100% boundary (ulong scaling precision), not a real
    # operating condition — clip them so they don't create a
    # spurious, undersampled "100_110" threshold bin.
    if "PWM" in clean.columns:
        overshoot_count = int((clean["PWM"] > 1.0).sum())
        if overshoot_count > 0:
            print(f"  Clipping {overshoot_count} PWM rows > 1.0 in {path.name}")
        clean["PWM"] = clean["PWM"].clip(upper=1.0)

    clean = clean.replace([np.inf, -np.inf], np.nan)
    clean = clean.dropna().reset_index(drop=True)

    if len(clean) < WINDOW_LEN:
        raise ValueError(f"{path.name} has only {len(clean)} valid rows.")

    return clean

In [ ]:
# ==========================================================
# CELL 5 — load labeled sessions
# ==========================================================
files = discover_files(BASE_PATH)

if not files:
    raise RuntimeError(
        f"No CSV or Excel files found under: {BASE_PATH}"
    )

sessions = []

for session_id, file_path in enumerate(files):
    label = label_for(file_path)

    if label is None:
        print(
            "SKIP unrecognized:",
            os.path.relpath(file_path, BASE_PATH),
        )
        continue

    try:
        df = read_one(file_path)
    except Exception as error:
        print(
            "SKIP read error:",
            os.path.relpath(file_path, BASE_PATH),
            "|",
            error,
        )
        continue

    class_id = LABEL_TO_ID[label]

    sessions.append({
        "session_id": session_id,
        "X": df.to_numpy(dtype=np.float32),
        "file": os.path.relpath(file_path, BASE_PATH),
        "label": label,
        "class_id": class_id,
        "anomaly": anomaly_target_from_class_id(class_id),
        "reconstruction_reference": int(is_normal_with_load_file(file_path, label)),
        "rows": len(df),
    })

if not sessions:
    raise RuntimeError(
        "No labeled sessions were loaded."
    )

class_counts = pd.Series([
    session["label"]
    for session in sessions
]).value_counts().reindex(
    FINAL_LABELS,
    fill_value=0,
)

print("\nFILES PER CLASS")
print(class_counts)

missing_classes = [
    label
    for label in FINAL_LABELS
    if class_counts[label] == 0
]

if missing_classes:
    raise RuntimeError(
        f"No files found for classes: {missing_classes}"
    )

loaded_normal_files = [
    session["file"]
    for session in sessions
    if session["reconstruction_reference"] == 1
]

unloaded_normal_files = [
    session["file"]
    for session in sessions
    if (
        session["label"] == "normal"
        and session["reconstruction_reference"] == 0
    )
]

print("\nNORMAL WITH LOAD — USED TO TRAIN THE RECONSTRUCTION AE")
for file_name in loaded_normal_files:
    print("  ", file_name)

print("\nNORMAL WITHOUT LOAD — EXCLUDED FROM RECONSTRUCTION TRAINING")
for file_name in unloaded_normal_files:
    print("  ", file_name)

if not loaded_normal_files:
    raise RuntimeError(
        "No normal-with-load files were identified."
    )

if not unloaded_normal_files:
    print(
        "WARNING: No normal-without-load files were identified. "
        "Check the filename terms in is_normal_with_load_file()."
    )

In [ ]:
# ==========================================================
# CELL 5.1 — (CHANGED) no canary hold-out.
# Per decision: 5-fold CV (CELL 12/13) is the validation signal;
# the final shipped model trains on 100% of sessions. The canary
# mechanism is removed so no session is excluded from CV or from
# final training.
# ==========================================================
CANARY_SESSIONS = []
TRAINING_SESSIONS = sessions  # 100% of data — used for both CV and final training

print("Canary hold-out disabled — using 100% of sessions for CV and final training.")
print("Training/CV sessions:", len(TRAINING_SESSIONS))

In [ ]:
# ==========================================================
# CELL 6 — create 30 × 14 windows
# ==========================================================
def make_windows_from_array(X_raw):
    starts = np.arange(
        0,
        len(X_raw) - WINDOW_LEN + 1,
        WINDOW_STEP,
        dtype=np.int64,
    )

    if len(starts) == 0:
        return None, None

    windows = np.stack([
        X_raw[start:start + WINDOW_LEN]
        for start in starts
    ]).astype(np.float32)

    return windows, starts


def build_dataset_from_sessions(selected_sessions):
    all_windows = []
    all_class_labels = []
    all_anomaly_labels = []
    all_files = []
    all_starts = []
    all_groups = []
    all_reconstruction_reference = []

    for session in selected_sessions:
        windows, starts = make_windows_from_array(
            session["X"]
        )

        if windows is None:
            continue

        number_of_windows = len(windows)

        all_windows.append(windows)

        all_class_labels.extend(
            [session["class_id"]] * number_of_windows
        )

        all_anomaly_labels.extend(
            [session["anomaly"]] * number_of_windows
        )

        all_files.extend(
            [session["file"]] * number_of_windows
        )

        all_starts.extend(
            starts.tolist()
        )

        all_groups.extend(
            [session["session_id"]] * number_of_windows
        )

        all_reconstruction_reference.extend(
            [session["reconstruction_reference"]] * number_of_windows
        )

    if not all_windows:
        raise RuntimeError(
            "No windows were created."
        )

    return (
        np.concatenate(
            all_windows,
            axis=0,
        ).astype(np.float32),
        np.asarray(
            all_class_labels,
            dtype=np.int64,
        ),
        np.asarray(
            all_anomaly_labels,
            dtype=np.float32,
        ),
        np.asarray(
            all_reconstruction_reference,
            dtype=np.float32,
        ),
        np.asarray(
            all_files,
            dtype=object,
        ),
        np.asarray(
            all_starts,
            dtype=np.int64,
        ),
        np.asarray(
            all_groups,
            dtype=np.int64,
        ),
        
    )

In [ ]:
# ==========================================================
# CELL 7 — normalization utilities
# ==========================================================
def calculate_normalization_statistics(X_train):
    flattened_rows = X_train.reshape(
        -1,
        N_FEATURES,
    )

    mean = np.mean(
        flattened_rows,
        axis=0,
    ).astype(np.float32)

    variance = np.var(
        flattened_rows,
        axis=0,
    ).astype(np.float32)

    variance = np.maximum(
        variance,
        1e-12,
    )

    return mean, variance


def normalize_windows_numpy(
    X,
    mean,
    variance,
):
    return (
        (
            X
            - mean.reshape(
                1,
                1,
                N_FEATURES,
            )
        )
        / np.sqrt(
            variance.reshape(
                1,
                1,
                N_FEATURES,
            )
            + 1e-7
        )
    ).astype(np.float32)

In [ ]:
# ==========================================================
# CELL 8 (UPDATED) — pure MLP ANN 3-head model with a DEDICATED,
# GRADIENT-ISOLATED encoder for reconstruction and logit-based anomaly output.
# ==========================================================
 
RECON_ENCODER_DENSE_1 = ENCODER_DENSE_1   # 32
RECON_ENCODER_DENSE_2 = ENCODER_DENSE_2   # 16
RECON_DROPOUT_1 = DROPOUT_LAYER_1         # 0.10
RECON_DROPOUT_2 = DROPOUT_LAYER_2         # 0.05
RECON_LATENT_UNITS = LATENT_DENSE_UNITS1   # 6
 
# Shared discriminative encoder (anomaly + classification only).
SHARED_ENCODER_DENSE_1 = ENCODER_DENSE_1
SHARED_ENCODER_DENSE_2 = ENCODER_DENSE_2
SHARED_DROPOUT_1 = DROPOUT_LAYER_1
SHARED_DROPOUT_2 = DROPOUT_LAYER_2
SHARED_LATENT_UNITS = LATENT_DENSE_UNITS2   #8
 
 
def window_mse_loss(y_true, y_pred):
    """Return one reconstruction MSE value per complete window."""
    squared_error = tf.square(y_true - y_pred)
    return tf.reduce_mean(squared_error, axis=[1, 2])
 
 
def build_hybrid_model(
    normalization_mean,
    normalization_variance,
):
    input_layer = tf.keras.Input(
        shape=(WINDOW_LEN, N_FEATURES),
        name="motor_window",
    )
 
    # Shared, fixed (non-trainable) normalization -- safe to share.
    normalizer = tf.keras.layers.Normalization(
        axis=-1,
        mean=normalization_mean,
        variance=normalization_variance,
        name="input_normalization",
    )
    normalized = normalizer(input_layer)
 
    flattened = tf.keras.layers.Flatten(
        name="flatten_window",
    )(normalized)
 
    # ------------------------------------------------------
    # BRANCH 1 — dedicated reconstruction encoder (isolated).
    # Only reconstruction_output's loss ever touches these layers.
    # ------------------------------------------------------
    recon_encoded = tf.keras.layers.Dense(
        RECON_ENCODER_DENSE_1, activation="relu", name="recon_encoder_dense1",
    )(flattened)
    recon_encoded = tf.keras.layers.Dropout(
        RECON_DROPOUT_1, name="recon_enc_dropout_1",
    )(recon_encoded)
    recon_encoded = tf.keras.layers.Dense(
        RECON_ENCODER_DENSE_2, activation="relu", name="recon_encoder_dense2",
    )(recon_encoded)
    recon_encoded = tf.keras.layers.Dropout(
        RECON_DROPOUT_2, name="recon_enc_dropout_2",
    )(recon_encoded)
    recon_latent = tf.keras.layers.Dense(
        RECON_LATENT_UNITS, activation="relu", name="recon_latent_dense",
    )(recon_encoded)
 
    decoded = tf.keras.layers.Dense(
        DECODER_DENSE_1, activation="relu", name="decoder_dense1",
    )(recon_latent)
    decoded = tf.keras.layers.Dense(
        DECODER_DENSE_2, activation="relu", name="decoder_dense2",
    )(decoded)
 
    reconstruction_flat = tf.keras.layers.Dense(
        WINDOW_LEN * N_FEATURES, activation="linear", name="reconstruction_dense",
    )(decoded)
 
    reconstruction_output = tf.keras.layers.Reshape(
        (WINDOW_LEN, N_FEATURES), name="reconstruction_output",
    )(reconstruction_flat)
 
    # ------------------------------------------------------
    # BRANCH 2 — shared discriminative encoder (anomaly + class only).
    # Completely separate weights from branch 1. Free to specialize
    # on fault-discriminative features without touching reconstruction.
    # ------------------------------------------------------
    shared_encoded = tf.keras.layers.Dense(
        SHARED_ENCODER_DENSE_1, activation="relu", name="shared_encoder_dense1",
    )(flattened)
    shared_encoded = tf.keras.layers.Dropout(
        SHARED_DROPOUT_1, name="shared_enc_dropout_1",
    )(shared_encoded)
    shared_encoded = tf.keras.layers.Dense(
        SHARED_ENCODER_DENSE_2, activation="relu", name="shared_encoder_dense2",
    )(shared_encoded)
    shared_encoded = tf.keras.layers.Dropout(
        SHARED_DROPOUT_2, name="shared_enc_dropout_2",
    )(shared_encoded)
    shared_latent = tf.keras.layers.Dense(
        SHARED_LATENT_UNITS, activation="relu", name="shared_latent_dense",
    )(shared_encoded)
 
    anomaly_features = tf.keras.layers.Dense(
        ANOMALY_DENSE_UNITS, activation="relu", name="anomaly_dense",
    )(shared_latent)
    
    # CHANGED: Removed activation="sigmoid" to output raw linear logits
    anomaly_output = tf.keras.layers.Dense(
        1, activation="linear", name="anomaly_output",
    )(anomaly_features)
 
    class_features = tf.keras.layers.Dense(
        CLASS_DENSE_UNITS, activation="relu", name="class_dense",
    )(shared_latent)
    class_output = tf.keras.layers.Dense(
        N_CLASSES, activation="softmax", name="class_output",
    )(class_features)
 
    model = tf.keras.Model(
        inputs=input_layer,
        outputs={
            "reconstruction_output": reconstruction_output,
            "anomaly_output": anomaly_output,
            "class_output": class_output,
        },
        name="dual_encoder_ae_anomaly_classifier",
    )
 
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss={
            "reconstruction_output": window_mse_loss,
            # CHANGED: Added from_logits=True to match the raw linear outputs safely
            "anomaly_output": tf.keras.losses.BinaryCrossentropy(from_logits=True),
            "class_output": tf.keras.losses.SparseCategoricalCrossentropy(),
        },
        loss_weights={
            "reconstruction_output": 1.00,
            "anomaly_output": ANOMALY_LOSS_WEIGHT,
            "class_output": CLASS_LOSS_WEIGHT,
        },
        metrics={
            "anomaly_output": [
                tf.keras.metrics.BinaryAccuracy(name="accuracy", threshold=0.0),
                tf.keras.metrics.AUC(name="auc", from_logits=True),
            ],
            "class_output": [
                tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy"),
            ],
        },
    )
 
    return model

In [ ]:
# ==========================================================
# CELL 9 (UPDATED) — training weights and data-split helpers
# (REWRITTEN: split_train_validation_by_group now isolates complete,
# whole-file paths chronologically to eliminate window data leakage)
# ==========================================================
def calculate_class_sample_weights(y_class, max_weight_ratio=4.0):
    present_classes = np.unique(y_class)

    class_weights = compute_class_weight(
        class_weight="balanced",
        classes=present_classes,
        y=y_class,
    )
    class_weights = np.clip(class_weights, None, max_weight_ratio * np.min(class_weights))

    lookup = {int(c): float(w) for c, w in zip(present_classes, class_weights)}

    return np.asarray([lookup[int(c)] for c in y_class], dtype=np.float32)


def calculate_anomaly_sample_weights(y_anomaly):
    y_integer = y_anomaly.astype(np.int64)
    present_classes = np.unique(y_integer)

    weights = compute_class_weight(
        class_weight="balanced",
        classes=present_classes,
        y=y_integer,
    )

    lookup = {
        int(class_id): float(weight)
        for class_id, weight in zip(
            present_classes,
            weights,
        )
    }

    return np.asarray([
        lookup[int(value)]
        for value in y_integer
    ], dtype=np.float32)


def make_training_sample_weights(
    y_class,
    y_anomaly,
    reconstruction_reference,
):
    reconstruction_weights = (
        reconstruction_reference.astype(np.float32)
    )

    class_weights = calculate_class_sample_weights(
        y_class
    )

    anomaly_weights = calculate_anomaly_sample_weights(
        y_anomaly
    )

    return {
        "reconstruction_output": reconstruction_weights,
        "anomaly_output": anomaly_weights,
        "class_output": class_weights,
    }


def split_train_validation_by_group(
    X,
    y_class,
    y_anomaly,
    reconstruction_reference,
    files,
    starts,
    groups,
):
    """
    CHANGED: Segregates training/validation splits strictly by WHOLE FILES.
    Extracts the string file path mapping array for each window, shuffles the 
    unique file identities, and maps 80% whole files to sub-training and 20% 
    whole files to sub-validation. This keeps time-series block windows 100% 
    sequential and isolated.
    """
    # Find the unique whole file-paths present in this cross-validation fold
    unique_files, first_indices = np.unique(files, return_index=True)
    file_labels = y_class[first_indices]

    class_file_counts = pd.Series(file_labels).value_counts()

    if class_file_counts.min() < 2:
        raise RuntimeError(
            "Each class needs at least 2 whole sessions inside the training fold "
            "to build a clean, leak-safe file-isolated validation split. "
            f"Current sub-fold file counts: {class_file_counts.to_dict()}"
        )

    # Calculate validation fraction safely at file level
    validation_fraction = max(0.15, N_CLASSES / len(unique_files))
    validation_fraction = min(validation_fraction, 0.40)

    # Perform the split strictly on unique FILE strings, not individual rows
    splitter = StratifiedShuffleSplit(
        n_splits=1,
        test_size=validation_fraction,
        random_state=SEED,
    )

    file_train_pos, file_validation_pos = next(
        splitter.split(unique_files, file_labels)
    )

    train_files_list = unique_files[file_train_pos]
    validation_files_list = unique_files[file_validation_pos]

    # Map the window-level table back to the assigned file masks
    train_index = np.flatnonzero(np.isin(files, train_files_list))
    validation_index = np.flatnonzero(np.isin(files, validation_files_list))

    if np.intersect1d(files[train_index], files[validation_index]).size:
        raise RuntimeError(
            "CRITICAL ERROR: File-level leakage detected between train and validation splits."
        )

    # Construct the chronological file datasets without scrambling the windows
    train_data = (
        X[train_index],
        y_class[train_index],
        y_anomaly[train_index],
        reconstruction_reference[train_index],
        files[train_index],
        starts[train_index],
        groups[train_index],
    )

    validation_data = (
        X[validation_index],
        y_class[validation_index],
        y_anomaly[validation_index],
        reconstruction_reference[validation_index],
        files[validation_index],
        starts[validation_index],
        groups[validation_index],
    )

    return train_data, validation_data

In [ ]:

# ==========================================================
# CELL 10 — Fit one fold/model
# Combined early stopping with best-weight restoration
# ==========================================================

RECON_STOPPING_WEIGHT = 0.10


class CombinedHeadEarlyStopping(tf.keras.callbacks.Callback):

    def __init__(
        self,
        patience=20,
        min_delta=1e-4,
        recon_weight=0.10,
    ):
        super().__init__()

        self.patience = patience
        self.min_delta = min_delta
        self.recon_weight = recon_weight

        self.best_score = -np.inf
        self.best_epoch = 0
        self.wait = 0
        self.best_weights = None
        self.stopped_epoch = 0

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}

        class_acc = logs.get(
            "val_class_output_accuracy", 0.0
        )

        recon_loss = logs.get(
            "val_reconstruction_output_loss", None
        )

        score = (
            class_acc
            if recon_loss is None
            else class_acc - self.recon_weight * recon_loss
        )

        if score > self.best_score + self.min_delta:

            self.best_score = score
            self.best_epoch = epoch + 1
            self.wait = 0

            self.best_weights = self.model.get_weights()

        else:
            self.wait += 1

            if self.wait >= self.patience:
                self.stopped_epoch = epoch + 1
                self.model.stop_training = True

    def on_train_end(self, logs=None):

        if self.best_weights is not None:
            self.model.set_weights(self.best_weights)

        print(
            f"Restored best validation weights. "
            f"Best epoch: {self.best_epoch}, "
            f"Best score: {self.best_score:.6f}"
        )


def fit_model_with_group_validation(
    X_train_full,
    y_class_full,
    y_anomaly_full,
    reconstruction_reference_full,
    files_full,
    starts_full,
    groups_full,
):

    (
        train_data,
        validation_data,
    ) = split_train_validation_by_group(
        X_train_full,
        y_class_full,
        y_anomaly_full,
        reconstruction_reference_full,
        files_full,
        starts_full,
        groups_full,
    )

    (
        X_train,
        y_class_train,
        y_anomaly_train,
        reconstruction_reference_train,
        _,
        _,
        _,
    ) = train_data

    (
        X_validation,
        y_class_validation,
        y_anomaly_validation,
        reconstruction_reference_validation,
        _,
        _,
        _,
    ) = validation_data

    normalization_mean, normalization_variance = (
        calculate_normalization_statistics(X_train)
    )

    X_train_normalized = normalize_windows_numpy(
        X_train,
        normalization_mean,
        normalization_variance,
    )

    X_validation_normalized = normalize_windows_numpy(
        X_validation,
        normalization_mean,
        normalization_variance,
    )

    model = build_hybrid_model(
        normalization_mean,
        normalization_variance,
    )

    sample_weights = make_training_sample_weights(
        y_class_train,
        y_anomaly_train,
        reconstruction_reference_train,
    )

    validation_sample_weights = {
        "reconstruction_output":
            reconstruction_reference_validation.astype(
                np.float32
            ),

        "anomaly_output":
            np.ones(
                len(y_anomaly_validation),
                dtype=np.float32,
            ),

        "class_output":
            np.ones(
                len(y_class_validation),
                dtype=np.float32,
            ),
    }

    callbacks = [
        CombinedHeadEarlyStopping(
            patience=20,
            min_delta=1e-4,
            recon_weight=RECON_STOPPING_WEIGHT,
        ),

        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_class_output_loss",
            mode="min",
            factor=0.5,
            patience=7,
            min_lr=1e-5,
            verbose=1,
        ),
    ]

    history = model.fit(
        X_train,
        {
            "reconstruction_output":
                X_train_normalized,

            "anomaly_output":
                y_anomaly_train,

            "class_output":
                y_class_train,
        },

        sample_weight=sample_weights,

        validation_data=(
            X_validation,
            {
                "reconstruction_output":
                    X_validation_normalized,

                "anomaly_output":
                    y_anomaly_validation,

                "class_output":
                    y_class_validation,
            },
            validation_sample_weights,
        ),

        epochs=MAX_EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=callbacks,
        verbose=2,
        shuffle=True,
    )

    return (
        model,
        normalization_mean,
        normalization_variance,
        history,
    )


In [ ]:
# ==========================================================
# CELL 11 — decision fusion and conditional thresholds
# ==========================================================
def get_pwm_bin_index(mean_pwm):
    bin_index = np.searchsorted(
        PWM_BIN_EDGES,
        mean_pwm,
        side="right",
    ) - 1

    return int(
        np.clip(
            bin_index,
            0,
            len(PWM_BIN_NAMES) - 1,
        )
    )


def get_conditional_reconstruction_threshold(
    predicted_class_id,
    mean_pwm,
    mean_irms,
):
    predicted_label = FINAL_LABELS[
        int(predicted_class_id)
    ]

    if predicted_label in [
        "motor_off",
        "turning_off",
        "turning_on",
    ]:
        return float(
            STATE_RECON_THRESHOLDS.get(
                predicted_label,
                RECON_THRESHOLD,
            )
        )

    bin_index = get_pwm_bin_index(mean_pwm)
    bin_name = PWM_BIN_NAMES[bin_index]

    return float(
        NORMAL_PWM_RECON_THRESHOLDS.get(
            bin_name,
            RECON_THRESHOLD,
        )
    )


def get_local_normal_baseline(mean_pwm):
    bin_index = get_pwm_bin_index(mean_pwm)
    bin_name = PWM_BIN_NAMES[bin_index]

    return float(
        LOCAL_NORMAL_BASELINES.get(
            bin_name,
            GLOBAL_NORMAL_BASELINE,
        )
    )


def severity_level_and_action(severity_ratio):
    if severity_ratio < SEVERITY_MILD_MULTIPLIER:
        return "normal", "No action needed."
    elif severity_ratio < SEVERITY_MODERATE_MULTIPLIER:
        return "mild", "Log and monitor; no immediate action."
    elif severity_ratio < SEVERITY_SEVERE_MULTIPLIER:
        return "moderate", "Schedule inspection soon."
    else:
        return "severe", "Stop motor / inspect immediately."


def evaluate_window(
    class_probabilities_row,
    anomaly_probability,
    reconstruction_mse,
    mean_pwm,
    mean_irms,
):
    class_id = int(np.argmax(class_probabilities_row))
    predicted_label = FINAL_LABELS[class_id]
    class_confidence = float(class_probabilities_row[class_id])

    binary_flag = bool(anomaly_probability >= ANOMALY_PROB_THRESHOLD)

    recon_threshold = get_conditional_reconstruction_threshold(
        class_id, mean_pwm, mean_irms,
    )
    recon_flag = bool(reconstruction_mse > recon_threshold)

    baseline = get_local_normal_baseline(mean_pwm)
    severity_ratio = (
        reconstruction_mse / baseline if baseline > 0 else 0.0
    )
    severity_label, recommended_action = severity_level_and_action(
        severity_ratio
    )

    is_known_fault = (
        class_id in FAULT_CLASS_IDS
        and class_confidence >= CLASS_CONF_THRESHOLD
    )

    if is_known_fault:
        final_label = predicted_label
    elif binary_flag or recon_flag:
        final_label = "unknown_fault"
    elif (
        class_id in SAFE_CLASS_IDS
        and class_confidence >= CLASS_CONF_THRESHOLD
    ):
        final_label = predicted_label
    else:
        final_label = "uncertain_safe"

    return {
        "final_label": final_label,
        "class_id": class_id,
        "class_confidence": class_confidence,
        "binary_flag": binary_flag,
        "recon_flag": recon_flag,
        "recon_threshold": recon_threshold,
        "local_baseline": baseline,
        "severity_ratio": severity_ratio,
        "severity_label": severity_label,
        "recommended_action": recommended_action,
    }


def fuse_decisions(
    class_probabilities,
    anomaly_probabilities,
    reconstruction_mse,
    reconstruction_thresholds,
):
    class_ids = np.argmax(
        class_probabilities,
        axis=1,
    ).astype(np.int64)

    class_confidence = np.max(
        class_probabilities,
        axis=1,
    )

    reconstruction_anomaly_flags = (
        reconstruction_mse
        > reconstruction_thresholds
    )

    binary_anomaly_flags = (
        anomaly_probabilities
        >= ANOMALY_PROB_THRESHOLD
    )

    confident_known_fault_flags = (
        np.isin(
            class_ids,
            FAULT_CLASS_IDS,
        )
        & (
            class_confidence
            >= CLASS_CONF_THRESHOLD
        )
    )

    anomaly_flags = (
        binary_anomaly_flags
        | reconstruction_anomaly_flags
        | confident_known_fault_flags
    )

    final_labels = []

    for (
        class_id,
        confidence,
        binary_flag,
        reconstruction_flag,
        known_fault_flag,
    ) in zip(
        class_ids,
        class_confidence,
        binary_anomaly_flags,
        reconstruction_anomaly_flags,
        confident_known_fault_flags,
    ):
        predicted_label = FINAL_LABELS[int(class_id)]

        if known_fault_flag:
            final_labels.append(predicted_label)
        elif binary_flag or reconstruction_flag:
            final_labels.append("unknown_fault")
        elif (
            class_id in SAFE_CLASS_IDS
            and confidence >= CLASS_CONF_THRESHOLD
        ):
            final_labels.append(predicted_label)
        else:
            final_labels.append("uncertain_safe")

    return (
        class_ids,
        class_confidence,
        binary_anomaly_flags,
        reconstruction_anomaly_flags,
        anomaly_flags,
        np.asarray(final_labels, dtype=object),
    )

In [ ]:

# ==========================================================
# CELL 12 — Five-fold session-wise cross-validation
# Correct anomaly logits and save detailed predictions
# ==========================================================

cv_table = pd.DataFrame({
    "label": [s["class_id"] for s in TRAINING_SESSIONS],
    "group": [s["session_id"] for s in TRAINING_SESSIONS],
})

splitter = StratifiedGroupKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED,
)

cv_metrics = []
all_true = []
all_pred = []
all_anomaly_true = []
all_anomaly_pred = []
all_reconstruction_mse = []
all_reconstruction_reference = []
all_mean_pwm = []
all_mean_irms = []
all_window_predictions = []
cv_best_epochs = []

pwm_idx = FEATURE_COLUMNS.index("PWM")
irms_idx = FEATURE_COLUMNS.index(LOAD_PROXY_FEATURE)
vib_rms_idx = FEATURE_COLUMNS.index("Vib_RMS")
vib_var_idx = FEATURE_COLUMNS.index("Vib_Variance")

for fold, (train_index, test_index) in enumerate(
    splitter.split(
        cv_table,
        cv_table["label"],
        groups=cv_table["group"],
    ),
    start=1,
):
    print("\n" + "=" * 70)
    print("FOLD", fold)
    print("=" * 70)

    train_sessions = [
        TRAINING_SESSIONS[i] for i in train_index
    ]
    test_sessions = [
        TRAINING_SESSIONS[i] for i in test_index
    ]

    (
        X_train_full,
        y_class_train_full,
        y_anomaly_train_full,
        reconstruction_reference_train_full,
        train_files,
        train_starts,
        train_groups,
    ) = build_dataset_from_sessions(train_sessions)

    (
        X_test,
        y_class_test,
        y_anomaly_test,
        reconstruction_reference_test,
        test_files,
        test_starts,
        test_groups,
    ) = build_dataset_from_sessions(test_sessions)

    # Verify complete session isolation
    assert not set(train_groups).intersection(
        set(test_groups)
    ), "Session leakage detected!"

    (
        model,
        normalization_mean,
        normalization_variance,
        history,
    ) = fit_model_with_group_validation(
        X_train_full,
        y_class_train_full,
        y_anomaly_train_full,
        reconstruction_reference_train_full,
        train_files,
        train_starts,
        train_groups,
    )

    validation_accuracy_history = history.history.get(
        "val_class_output_accuracy", []
    )

    if validation_accuracy_history:
        best_epoch = int(
            np.argmax(validation_accuracy_history) + 1
        )
    else:
        best_epoch = len(history.history.get("loss", []))

    cv_best_epochs.append(best_epoch)
    print("Best validation epoch:", best_epoch)

    # Normalization reference for reconstruction MSE
    X_test_normalized = normalize_windows_numpy(
        X_test,
        normalization_mean,
        normalization_variance,
    )

    prediction_dict = model.predict(
        X_test,
        batch_size=BATCH_SIZE,
        verbose=0,
    )

    reconstructed = prediction_dict[
        "reconstruction_output"
    ]

    class_probabilities = prediction_dict[
        "class_output"
    ]

    # IMPORTANT: anomaly head outputs RAW LOGITS.
    anomaly_logits = prediction_dict[
        "anomaly_output"
    ].reshape(-1)

    anomaly_probabilities = tf.math.sigmoid(
        anomaly_logits
    ).numpy().reshape(-1)

    # Classification
    class_predictions = np.argmax(
        class_probabilities, axis=1
    ).astype(np.int64)

    class_confidence = np.max(
        class_probabilities, axis=1
    )

    anomaly_predictions = (
        anomaly_probabilities >= ANOMALY_PROB_THRESHOLD
    ).astype(np.int64)

    # Reconstruction MSE (normalized feature space)
    reconstruction_mse = np.mean(
        np.square(reconstructed - X_test_normalized),
        axis=(1, 2),
    )

    # Window-level physical features
    test_window_mean_pwm = X_test[
        :, :, pwm_idx
    ].mean(axis=1).astype(np.float32)

    test_window_mean_irms = X_test[
        :, :, irms_idx
    ].mean(axis=1).astype(np.float32)

    test_window_mean_vib_rms = X_test[
        :, :, vib_rms_idx
    ].mean(axis=1).astype(np.float32)

    test_window_mean_vib_variance = X_test[
        :, :, vib_var_idx
    ].mean(axis=1).astype(np.float32)

    # Store pooled CV results used by Cell 13
    all_true.extend(y_class_test.tolist())
    all_pred.extend(class_predictions.tolist())

    all_anomaly_true.extend(
        y_anomaly_test.astype(np.int64).tolist()
    )
    all_anomaly_pred.extend(
        anomaly_predictions.tolist()
    )

    all_reconstruction_mse.extend(
        reconstruction_mse.tolist()
    )
    all_reconstruction_reference.extend(
        reconstruction_reference_test.astype(
            np.int64
        ).tolist()
    )
    all_mean_pwm.extend(
        test_window_mean_pwm.tolist()
    )
    all_mean_irms.extend(
        test_window_mean_irms.tolist()
    )

    # Fold evaluation
    fold_metrics = {
        "fold": fold,
        "accuracy": accuracy_score(
            y_class_test, class_predictions
        ),
        "balanced_accuracy": balanced_accuracy_score(
            y_class_test, class_predictions
        ),
        "macro_precision": precision_score(
            y_class_test,
            class_predictions,
            average="macro",
            zero_division=0,
        ),
        "macro_recall": recall_score(
            y_class_test,
            class_predictions,
            average="macro",
            zero_division=0,
        ),
        "macro_f1": f1_score(
            y_class_test,
            class_predictions,
            average="macro",
            zero_division=0,
        ),
        "anomaly_accuracy": accuracy_score(
            y_anomaly_test,
            anomaly_predictions,
        ),
        "anomaly_f1": f1_score(
            y_anomaly_test,
            anomaly_predictions,
            zero_division=0,
        ),
        "mean_reconstruction_mse": float(
            np.mean(reconstruction_mse)
        ),
    }

    cv_metrics.append(fold_metrics)

    # Detailed held-out predictions
    fold_predictions = pd.DataFrame({
        "fold": fold,
        "session_id": test_groups,
        "file": test_files,
        "start_row": test_starts,
        "mean_pwm": test_window_mean_pwm,
        "mean_irms": test_window_mean_irms,
        "mean_vib_rms": test_window_mean_vib_rms,
        "mean_vib_variance":
            test_window_mean_vib_variance,
        "true_class_id": y_class_test,
        "true_label": [
            FINAL_LABELS[int(i)] for i in y_class_test
        ],
        "pred_class_id": class_predictions,
        "pred_label": [
            FINAL_LABELS[int(i)]
            for i in class_predictions
        ],
        "class_confidence": class_confidence,
        "true_anomaly":
            y_anomaly_test.astype(np.int64),
        "reconstruction_reference":
            reconstruction_reference_test.astype(np.int64),
        "anomaly_logit": anomaly_logits,
        "anomaly_probability": anomaly_probabilities,
        "anomaly_prediction": anomaly_predictions,
        "reconstruction_mse": reconstruction_mse,
    })

    for class_id, label in enumerate(FINAL_LABELS):
        fold_predictions[
            f"probability_{label}"
        ] = class_probabilities[:, class_id]

    all_window_predictions.append(fold_predictions)

    print(pd.Series(fold_metrics))

    print("\nFOLD CLASSIFICATION REPORT")
    print(
        classification_report(
            y_class_test,
            class_predictions,
            labels=np.arange(N_CLASSES),
            target_names=FINAL_LABELS,
            zero_division=0,
        )
    )

    # Confusion matrix
    fold_cm = confusion_matrix(
        y_class_test,
        class_predictions,
        labels=np.arange(N_CLASSES),
    )

    pd.DataFrame(
        fold_cm,
        index=FINAL_LABELS,
        columns=FINAL_LABELS,
    ).to_csv(
        os.path.join(
            CV_DIR,
            f"fold_{fold}_confusion_matrix.csv",
        )
    )

    fig, axis = plt.subplots(figsize=(9, 8))
    ConfusionMatrixDisplay(
        fold_cm,
        display_labels=FINAL_LABELS,
    ).plot(
        ax=axis,
        values_format="d",
        xticks_rotation=45,
        colorbar=False,
    )
    axis.set_title(
        f"Fold {fold} — Session-wise Test Confusion Matrix"
    )
    fig.tight_layout()
    fig.savefig(
        os.path.join(
            CV_DIR,
            f"fold_{fold}_confusion_matrix.png",
        ),
        dpi=180,
    )
    plt.show()
    plt.close(fig)

    history_df = pd.DataFrame(history.history)
    history_df.to_csv(
        os.path.join(
            CV_DIR,
            f"fold_{fold}_training_history.csv",
        ),
        index=False,
    )

    tf.keras.backend.clear_session()


In [ ]:
# ==========================================================
# CELL 13 — combined CV report, confidence-aware thresholds,
# and swept operating points
# (CHANGED: 1) small-sample quantile thresholds are inflated by
# their own bootstrap uncertainty instead of being trusted blindly,
# and a confidence value is stored alongside every threshold;
# 2) ANOMALY_PROB_THRESHOLD and CLASS_CONF_THRESHOLD are now
# picked from a precision/recall sweep over held-out CV
# predictions instead of being fixed constants.)
# ==========================================================
from sklearn.metrics import precision_recall_curve

ANOMALY_TARGET_RECALL = 0.95          # must catch >=95% of true faults
CLASS_CONF_TARGET_ACCURACY = 0.95     # confident predictions must be >=95% correct
MIN_SAMPLES_FOR_FULL_CONFIDENCE = 100


def quantile_with_confidence(errors, quantile, min_full_confidence=MIN_SAMPLES_FOR_FULL_CONFIDENCE, n_boot=300, seed=SEED):
    """Quantile threshold, inflated by its own bootstrap uncertainty
    when the sample count is small. Returns (threshold, confidence in [0,1])."""
    errors = np.asarray(errors, dtype=np.float64)
    n = len(errors)

    if n == 0:
        return None, 0.0

    raw_threshold = float(np.quantile(errors, quantile))

    if n >= min_full_confidence:
        return raw_threshold, 1.0

    rng = np.random.default_rng(seed)
    boot_quantiles = [
        np.quantile(rng.choice(errors, size=n, replace=True), quantile)
        for _ in range(n_boot)
    ]
    boot_std = float(np.std(boot_quantiles))
    confidence = n / min_full_confidence
    inflated_threshold = raw_threshold + (1 - confidence) * boot_std

    return inflated_threshold, confidence


cv_metrics_df = pd.DataFrame(
    cv_metrics
)

cv_predictions_df = pd.concat(
    all_window_predictions,
    ignore_index=True,
)

cv_metrics_df.to_csv(
    os.path.join(
        CV_DIR,
        "cv_metrics.csv",
    ),
    index=False,
)

cv_predictions_df.to_csv(
    os.path.join(
        CV_DIR,
        "cv_window_predictions.csv",
    ),
    index=False,
)

all_true = np.asarray(all_true, dtype=np.int64)
all_pred = np.asarray(all_pred, dtype=np.int64)
all_anomaly_true = np.asarray(all_anomaly_true, dtype=np.int64)
all_anomaly_pred = np.asarray(all_anomaly_pred, dtype=np.int64)
all_mean_pwm = np.asarray(all_mean_pwm, dtype=np.float32)
all_mean_irms = np.asarray(all_mean_irms, dtype=np.float32)
all_reconstruction_mse = np.asarray(all_reconstruction_mse, dtype=np.float32)
all_reconstruction_reference = np.asarray(all_reconstruction_reference, dtype=np.int64)

# ==========================================================
# SWEEP 1 — anomaly probability threshold from a PR curve over
# all held-out CV window predictions.
# ==========================================================
precision_curve, recall_curve, anomaly_sweep_thresholds = precision_recall_curve(
    cv_predictions_df["true_anomaly"],
    cv_predictions_df["anomaly_probability"],
)

anomaly_sweep_df = pd.DataFrame({
    "threshold": np.append(anomaly_sweep_thresholds, 1.0),
    "precision": precision_curve,
    "recall": recall_curve,
})
anomaly_sweep_df.to_csv(
    os.path.join(CV_DIR, "anomaly_threshold_sweep.csv"),
    index=False,
)

valid_recall_idx = np.where(recall_curve[:-1] >= ANOMALY_TARGET_RECALL)[0]

if len(valid_recall_idx):
    best_idx = valid_recall_idx[np.argmax(precision_curve[valid_recall_idx])]
    ANOMALY_PROB_THRESHOLD = float(anomaly_sweep_thresholds[best_idx])
else:
    fallback_idx = int(np.argmax(recall_curve[:-1]))
    ANOMALY_PROB_THRESHOLD = float(anomaly_sweep_thresholds[fallback_idx])
    print(
        f"WARNING: no threshold reaches the {ANOMALY_TARGET_RECALL:.2f} "
        "target recall on CV data; using the highest-recall threshold found."
    )

print(f"\nSwept ANOMALY_PROB_THRESHOLD -> {ANOMALY_PROB_THRESHOLD:.4f}")

# ==========================================================
# SWEEP 2 — class confidence threshold from an accuracy/coverage
# sweep over all held-out CV window predictions.
# ==========================================================
def sweep_class_confidence_threshold(df, target_accuracy, thresholds=np.linspace(0.30, 0.95, 66)):
    rows = []
    for t in thresholds:
        subset = df[df["class_confidence"] >= t]
        if len(subset) == 0:
            continue
        acc = float((subset["true_class_id"] == subset["pred_class_id"]).mean())
        coverage = len(subset) / len(df)
        rows.append((t, acc, coverage))

    sweep_df = pd.DataFrame(rows, columns=["threshold", "accuracy", "coverage"])
    passing = sweep_df[sweep_df["accuracy"] >= target_accuracy]

    if len(passing):
        chosen = float(passing.sort_values("threshold").iloc[0]["threshold"])
    else:
        chosen = float(sweep_df.sort_values("accuracy", ascending=False).iloc[0]["threshold"])
        print(
            f"WARNING: no threshold reaches {target_accuracy:.2f} confident-accuracy "
            "on CV data; using the highest-accuracy threshold found."
        )

    return chosen, sweep_df


CLASS_CONF_THRESHOLD, class_conf_sweep_df = sweep_class_confidence_threshold(
    cv_predictions_df, CLASS_CONF_TARGET_ACCURACY,
)

class_conf_sweep_df.to_csv(
    os.path.join(CV_DIR, "class_confidence_threshold_sweep.csv"),
    index=False,
)

print(f"Swept CLASS_CONF_THRESHOLD -> {CLASS_CONF_THRESHOLD:.4f}")

# ==========================================================
# HELD-OUT SAFE THRESHOLDS (confidence-aware quantiles):
#   1) state-specific thresholds for motor_off / transitions
#   2) one dedicated PWM-bin threshold for steady normal,
#      calculated ONLY from held-out normal-WITH-LOAD windows
# ==========================================================
NORMAL_CLASS_ID = LABEL_TO_ID["normal"]

loaded_normal_mask = (
    all_reconstruction_reference == 1
)

if not np.any(loaded_normal_mask):
    raise RuntimeError(
        "No held-out normal-with-load windows are available "
        "for reconstruction-threshold calibration."
    )

loaded_normal_reconstruction_mse = (
    all_reconstruction_mse[loaded_normal_mask]
)

RECON_THRESHOLD, RECON_THRESHOLD_CONFIDENCE = quantile_with_confidence(
    loaded_normal_reconstruction_mse, RECON_QUANTILE,
)

GLOBAL_NORMAL_BASELINE = float(
    np.median(loaded_normal_reconstruction_mse)
)

STATE_RECON_THRESHOLDS = {}
STATE_THRESHOLD_CONFIDENCE = {}

for state_label in [
    "motor_off",
    "turning_off",
    "turning_on",
]:
    state_class_id = LABEL_TO_ID[state_label]
    state_errors = all_reconstruction_mse[all_true == state_class_id]

    if len(state_errors) >= MIN_THRESHOLD_WINDOWS:
        state_threshold, state_confidence = quantile_with_confidence(
            state_errors, RECON_QUANTILE,
        )
    else:
        state_threshold, state_confidence = RECON_THRESHOLD, 0.0
        print(
            f"WARNING: state {state_label} has only "
            f"{len(state_errors)} held-out windows. "
            "Using global safe fallback threshold."
        )

    STATE_RECON_THRESHOLDS[state_label] = state_threshold
    STATE_THRESHOLD_CONFIDENCE[state_label] = state_confidence

    print(
        f"State {state_label} | windows={len(state_errors)} | "
        f"threshold={state_threshold:.8f} | confidence={state_confidence:.2f}"
    )


normal_mask = loaded_normal_mask

if not np.any(normal_mask):
    raise RuntimeError(
        "No held-out steady normal-with-load windows are available "
        "for PWM threshold calibration."
    )

normal_reconstruction_mse = all_reconstruction_mse[normal_mask]
normal_mean_pwm = all_mean_pwm[normal_mask]

GLOBAL_NORMAL_THRESHOLD, _ = quantile_with_confidence(
    normal_reconstruction_mse, RECON_QUANTILE,
)

NORMAL_PWM_RECON_THRESHOLDS = {}
LOCAL_NORMAL_BASELINES = {}
PWM_THRESHOLD_CONFIDENCE = {}
threshold_rows = []

for bin_index, bin_name in enumerate(PWM_BIN_NAMES):
    lower_pwm = PWM_BIN_EDGES[bin_index]
    upper_pwm = PWM_BIN_EDGES[bin_index + 1]

    pwm_mask = (normal_mean_pwm >= lower_pwm) & (normal_mean_pwm < upper_pwm)
    pwm_errors = normal_reconstruction_mse[pwm_mask]

    if len(pwm_errors) >= MIN_THRESHOLD_WINDOWS:
        pwm_threshold, pwm_confidence = quantile_with_confidence(
            pwm_errors, RECON_QUANTILE,
        )
        pwm_baseline = float(np.median(pwm_errors))
    else:
        pwm_threshold, pwm_confidence = GLOBAL_NORMAL_THRESHOLD, 0.0
        pwm_baseline = GLOBAL_NORMAL_BASELINE
        print(
            f"WARNING: PWM bin {bin_name} has only "
            f"{len(pwm_errors)} normal windows. "
            "Using global normal fallback threshold."
        )

    NORMAL_PWM_RECON_THRESHOLDS[bin_name] = pwm_threshold
    LOCAL_NORMAL_BASELINES[bin_name] = pwm_baseline
    PWM_THRESHOLD_CONFIDENCE[bin_name] = pwm_confidence

    threshold_rows.append({
        "operating_state": "normal",
        "pwm_bin": bin_name,
        "lower_pwm": float(lower_pwm),
        "upper_pwm": float(upper_pwm),
        "normal_windows": int(len(pwm_errors)),
        "reconstruction_threshold": pwm_threshold,
        "local_baseline": pwm_baseline,
        "threshold_confidence": pwm_confidence,
    })

    ratio = pwm_threshold / pwm_baseline if pwm_baseline > 0 else float("nan")
    print(
        f"Normal {bin_name} | windows={len(pwm_errors)} | "
        f"threshold={pwm_threshold:.8f} | baseline={pwm_baseline:.8f} | "
        f"ratio={ratio:.2f} | confidence={pwm_confidence:.2f}"
    )

threshold_table = pd.DataFrame(threshold_rows)
threshold_table.to_csv(
    os.path.join(CV_DIR, "conditional_reconstruction_thresholds.csv"),
    index=False,
)

print("\nGlobal safe reconstruction fallback:", RECON_THRESHOLD)

overall_metrics = {
    "accuracy": accuracy_score(all_true, all_pred),
    "balanced_accuracy": balanced_accuracy_score(all_true, all_pred),
    "macro_precision": precision_score(all_true, all_pred, average="macro", zero_division=0),
    "macro_recall": recall_score(all_true, all_pred, average="macro", zero_division=0),
    "macro_f1": f1_score(all_true, all_pred, average="macro", zero_division=0),
    "anomaly_accuracy": accuracy_score(all_anomaly_true, all_anomaly_pred),
    "anomaly_f1": f1_score(all_anomaly_true, all_anomaly_pred, zero_division=0),
    "reconstruction_threshold": RECON_THRESHOLD,
    "swept_anomaly_prob_threshold": ANOMALY_PROB_THRESHOLD,
    "swept_class_conf_threshold": CLASS_CONF_THRESHOLD,
}

print("\nOVERALL CROSS-VALIDATION RESULTS")
print(pd.Series(overall_metrics))

print(
    classification_report(
        all_true,
        all_pred,
        labels=np.arange(N_CLASSES),
        target_names=FINAL_LABELS,
        zero_division=0,
    )
)

pd.DataFrame([overall_metrics]).to_csv(
    os.path.join(CV_DIR, "overall_metrics.csv"),
    index=False,
)

ConfusionMatrixDisplay(
    confusion_matrix(all_true, all_pred, labels=np.arange(N_CLASSES)),
    display_labels=FINAL_LABELS,
).plot(values_format="d", xticks_rotation=45)

plt.title("Pure MLP ANN 3-Head Hybrid Cross-Validation")
plt.tight_layout()
plt.savefig(os.path.join(CV_DIR, "confusion_matrix.png"), dpi=180)
plt.show()

In [ ]:

# ==========================================================
# CELL 13A — PWM REGION ERROR DIAGNOSTICS
# ==========================================================

diag = cv_predictions_df.copy()

required_columns = [
    "session_id", "file", "start_row",
    "mean_pwm", "mean_irms",
    "mean_vib_rms", "mean_vib_variance",
    "true_label", "pred_label",
    "true_anomaly", "pred_class_id",
    "class_confidence", "anomaly_probability",
    "reconstruction_mse",
]

missing = [
    c for c in required_columns
    if c not in diag.columns
]

if missing:
    raise RuntimeError(
        f"Missing columns: {missing}. "
        "Run the replacement Cell 12 and Cell 13."
    )

# IMPORTANT:
# Cell 12 already converted logits to probabilities.
# Do NOT apply sigmoid again.

diag["pwm_percent"] = diag["mean_pwm"] * 100.0

diag["pwm_region"] = np.select(
    [
        diag["pwm_percent"].between(28, 35),
        diag["pwm_percent"].between(60, 75),
    ],
    [
        "28-35% PWM",
        "60-75% PWM",
    ],
    default="Other PWM",
)

diag["classifier_error"] = (
    diag["true_label"] != diag["pred_label"]
)

diag["binary_flag"] = (
    diag["anomaly_probability"]
    >= ANOMALY_PROB_THRESHOLD
)

diag["reconstruction_threshold"] = [
    get_conditional_reconstruction_threshold(
        class_id, pwm, irms
    )
    for class_id, pwm, irms in zip(
        diag["pred_class_id"],
        diag["mean_pwm"],
        diag["mean_irms"],
    )
]

diag["recon_flag"] = (
    diag["reconstruction_mse"]
    > diag["reconstruction_threshold"]
)

diag["known_fault_flag"] = (
    diag["pred_class_id"].isin(FAULT_CLASS_IDS)
    & (
        diag["class_confidence"]
        >= CLASS_CONF_THRESHOLD
    )
)

diag["final_anomaly"] = (
    diag["binary_flag"]
    | diag["recon_flag"]
    | diag["known_fault_flag"]
)

diag["true_anomaly"] = diag[
    "true_anomaly"
].astype(bool)

diag["false_alarm"] = (
    ~diag["true_anomaly"]
    & diag["final_anomaly"]
)

diag["missed_fault"] = (
    diag["true_anomaly"]
    & ~diag["final_anomaly"]
)

def summarize_region(df):
    healthy = df[~df["true_anomaly"]]
    faults = df[df["true_anomaly"]]

    mechanical = df[
        df["true_label"] == "mechanical_fault"
    ]

    return pd.Series({
        "windows": len(df),
        "sessions": df["session_id"].nunique(),

        "healthy_windows": len(healthy),
        "fault_windows": len(faults),

        "mean_irms": df["mean_irms"].mean(),
        "mean_vib_rms": df["mean_vib_rms"].mean(),
        "mean_vib_variance":
            df["mean_vib_variance"].mean(),

        "classifier_error_pct":
            df["classifier_error"].mean() * 100
            if len(df) else np.nan,

        "healthy_binary_alarm_pct":
            healthy["binary_flag"].mean() * 100
            if len(healthy) else np.nan,

        "healthy_recon_alarm_pct":
            healthy["recon_flag"].mean() * 100
            if len(healthy) else np.nan,

        "healthy_final_false_alarm_pct":
            healthy["final_anomaly"].mean() * 100
            if len(healthy) else np.nan,

        "fault_detection_pct":
            faults["final_anomaly"].mean() * 100
            if len(faults) else np.nan,

        "mechanical_fault_detection_pct":
            mechanical["final_anomaly"].mean() * 100
            if len(mechanical) else np.nan,

        "missed_fault_windows":
            int(df["missed_fault"].sum()),
    })


regions = [
    "28-35% PWM",
    "60-75% PWM",
    "Other PWM",
]

region_summary = pd.DataFrame({
    region: summarize_region(
        diag[diag["pwm_region"] == region]
    )
    for region in regions
}).T

# Normal WITH LOAD only
normal_load = diag[
    (diag["true_label"] == "normal")
    & (diag["reconstruction_reference"] == 1)
].copy()

normal_load_summary = pd.DataFrame({
    region: summarize_region(
        normal_load[
            normal_load["pwm_region"] == region
        ]
    )
    for region in regions
}).T

# Session-specific diagnostics
session_rows = []

for (sid, file_name, region), group in diag.groupby(
    ["session_id", "file", "pwm_region"]
):
    result = summarize_region(group).to_dict()
    result.update({
        "session_id": sid,
        "file": file_name,
        "pwm_region": region,
        "true_label": group["true_label"].iloc[0],
    })
    session_rows.append(result)

session_summary = pd.DataFrame(session_rows)

# Save reports
reports = {
    "step1_window_diagnostics.csv": diag,
    "step1_pwm_region_summary.csv": region_summary,
    "step1_normal_with_load_summary.csv":
        normal_load_summary,
    "step1_session_region_summary.csv":
        session_summary,
}

for filename, frame in reports.items():
    frame.to_csv(
        os.path.join(CV_DIR, filename),
        index=False,
    )
    print("Saved:", filename)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

print("\n" + "=" * 70)
print("STEP 1 — PWM REGION ERROR ANALYSIS")
print("=" * 70)
print(region_summary.round(3).to_string())

print("\n" + "=" * 70)
print("NORMAL WITH LOAD — FALSE ALARM ANALYSIS")
print("=" * 70)
print(normal_load_summary.round(3).to_string())

print("\nAnomaly threshold:",
      ANOMALY_PROB_THRESHOLD)

print("\nSTEP 1 COMPLETE")
print(
    "NOTE: Fused-decision metrics are exploratory because "
    "Cell 13 calibrated thresholds on pooled CV predictions."
)


In [ ]:

# ==========================================================
# CELL 13B — Check live-observed fault confusions offline
# ==========================================================

normal = diag[
    (diag["true_label"] == "normal")
    & (diag["reconstruction_reference"] == 1)
].copy()

checks = [
    ("28-35% PWM", "electrical_fault"),
    ("60-70% PWM", "mechanical_fault"),
]

for region, fault_label in checks:

    if region == "28-35% PWM":
        subset = normal[
            normal["pwm_percent"].between(28, 35)
        ]
    else:
        subset = normal[
            normal["pwm_percent"].between(60, 80)
        ]

    wrong = subset[
        subset["pred_label"] == fault_label
    ]

    print("\n" + "=" * 65)
    print("Region:", region)
    print("Incorrect fault:", fault_label)
    print("Healthy windows:", len(subset))
    print("Incorrectly classified:", len(wrong))

    if len(subset):
        print(
            "Misclassification rate:",
            f"{100 * len(wrong) / len(subset):.2f}%"
        )

    if len(wrong):
        print("\nAffected sessions:")
        print(
            wrong.groupby(["session_id", "file"])
            .size()
            .sort_values(ascending=False)
            .to_string()
        )

        print("\nMean measurements:")
        print(
            wrong[
                [
                    "mean_pwm",
                    "mean_irms",
                    "mean_vib_rms",
                    "mean_vib_variance",
                    "class_confidence",
                    "anomaly_probability",
                    "reconstruction_mse",
                ]
            ].mean().round(5).to_string()
        )

        print("\nIndividual fault predictions:")
        print(
            wrong[
                [
                    "session_id",
                    "start_row",
                    "mean_pwm",
                    "mean_irms",
                    "pred_label",
                    "class_confidence",
                    "anomaly_probability",
                ]
            ].head(15).to_string(index=False)
        )

print("\nCELL 13B COMPLETE")


In [ ]:
# ==========================================================
# CELL 14 — train final model on all TRAINING sessions

# ==========================================================
(
    X_all,
    y_class_all,
    y_anomaly_all,
    reconstruction_reference_all,
    all_files,
    all_starts,
    all_groups,
) = build_dataset_from_sessions(
    TRAINING_SESSIONS
)

final_mean, final_variance = calculate_normalization_statistics(
    X_all
)

X_all_normalized = normalize_windows_numpy(
    X_all,
    final_mean,
    final_variance,
)

final_model = build_hybrid_model(
    final_mean,
    final_variance,
)

final_sample_weights = make_training_sample_weights(
    y_class_all,
    y_anomaly_all,
    reconstruction_reference_all,
)

FINAL_EPOCHS = int(
    np.median(cv_best_epochs)
)
FINAL_EPOCHS = max(FINAL_EPOCHS, 1)

print("\n" + "=" * 70)
print("FINAL DEPLOYMENT TRAINING ON TRAINING_SESSIONS (canary excluded)")
print("=" * 70)
print("Sessions used:", len(np.unique(all_groups)))
print("Windows used:", len(X_all))
print("Final epochs from median CV best epoch:", FINAL_EPOCHS)

final_history = final_model.fit(
    X_all,
    {
        "reconstruction_output": X_all_normalized,
        "anomaly_output": y_anomaly_all,
        "class_output": y_class_all,
    },
    sample_weight=final_sample_weights,
    epochs=FINAL_EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=2,
    shuffle=True,
)

final_prediction_dict = final_model.predict(
    X_all,
    batch_size=BATCH_SIZE,
    verbose=0,
)

final_class_probabilities = final_prediction_dict["class_output"]
final_class_predictions = np.argmax(final_class_probabilities, axis=1)

print("\nFINAL TRAINING-DATA CLASSIFICATION REPORT")
print(
    classification_report(
        y_class_all,
        final_class_predictions,
        labels=np.arange(N_CLASSES),
        target_names=FINAL_LABELS,
        zero_division=0,
    )
)
print("Final training-data accuracy:", accuracy_score(y_class_all, final_class_predictions))
print(
    "NOTE: this is training-set fit, not an unbiased generalization result. "
    "Use the cross-validation metrics AND the CELL 14B canary result for "
    "real model performance."
)

pd.DataFrame(final_history.history).to_csv(
    os.path.join(FINAL_DIR, "final_training_history.csv"),
    index=False,
)

In [ ]:
# ==========================================================
# CELL 14.1 — (CHANGED) canary gate removed.
# No held-out canary set exists anymore — the final model is
# trained on 100% of sessions (CELL 14 already uses
# TRAINING_SESSIONS, which now equals `sessions`). The honest
# generalization estimate is the 5-fold CV report in CELL 13.
# ==========================================================
print("Canary sanity gate skipped — model trained on 100% of sessions.")
print("See CELL 13 ('OVERALL CROSS-VALIDATION RESULTS') for the honest generalization estimate.")

In [ ]:
# ==========================================================
# CELL 15 — save model, labels, metadata, thresholds
# ==========================================================
KERAS_PATH = os.path.abspath(
    os.path.join(
        FINAL_DIR,
        "pure_mlp_ae_anomaly_classifier.keras",
    )
)

H5_PATH = os.path.abspath(
    os.path.join(
        FINAL_DIR,
        "pure_mlp_ae_anomaly_classifier.h5",
    )
)

LABELS_PATH = os.path.abspath(
    os.path.join(
        FINAL_DIR,
        "labels.txt",
    )
)

METADATA_PATH = os.path.abspath(
    os.path.join(
        FINAL_DIR,
        "metadata.json",
    )
)

THRESHOLDS_PATH = os.path.abspath(
    os.path.join(
        FINAL_DIR,
        "thresholds.json",
    )
)

final_model.save(
    KERAS_PATH
)

final_model.save(
    H5_PATH
)

with open(
    LABELS_PATH,
    "w",
    encoding="utf-8",
) as file:
    file.write(
        "\n".join(FINAL_LABELS)
    )

thresholds = {
    "global_reconstruction_fallback":
        RECON_THRESHOLD,

    "global_normal_baseline":
        GLOBAL_NORMAL_BASELINE,

    "state_reconstruction_thresholds":
        STATE_RECON_THRESHOLDS,

    "pwm_bin_edges":
        PWM_BIN_EDGES.tolist(),

    "pwm_bin_names":
        PWM_BIN_NAMES,

    "load_proxy_feature":
        LOAD_PROXY_FEATURE,

    "normal_pwm_reconstruction_thresholds":
        NORMAL_PWM_RECON_THRESHOLDS,

    "local_normal_baselines":
        LOCAL_NORMAL_BASELINES,

    "anomaly_probability_threshold":
        ANOMALY_PROB_THRESHOLD,

    "class_confidence_threshold":
        CLASS_CONF_THRESHOLD,
}

metadata = {
    "model_name":
        "pure_mlp_ae_anomaly_classifier",
    "input_shape": [
        WINDOW_LEN,
        N_FEATURES,
    ],
    "feature_columns":
        FEATURE_COLUMNS,
    "labels":
        FINAL_LABELS,
    "safe_labels":
        SAFE_LABELS,
    "fault_labels":
        FAULT_LABELS,
    "scale_divisor":
        SCALE_DIVISOR,
    "window_step":
        WINDOW_STEP,
    "row_period_sec":
        ROW_PERIOD_SEC,
    "parameter_count":
        int(
            final_model.count_params()
        ),
    "normalization_mean":
        final_mean.tolist(),
    "normalization_variance":
        final_variance.tolist(),
    "final_training_epochs":
        FINAL_EPOCHS,
    "thresholds":
        thresholds,
}

with open(
    METADATA_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        metadata,
        file,
        indent=2,
    )

with open(
    THRESHOLDS_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        thresholds,
        file,
        indent=2,
    )

print("Saved:", KERAS_PATH)
print("Saved:", H5_PATH)
print("Saved:", LABELS_PATH)
print("Saved:", METADATA_PATH)
print("Saved:", THRESHOLDS_PATH)

In [ ]:
# ==========================================================
# CELL 16 — export ONNX
# ==========================================================
import tf2onnx

ONNX_PATH = os.path.abspath(
    os.path.join(
        FINAL_DIR,
        "pure_mlp_ae_anomaly_classifier.onnx",
    )
)

input_signature = (
    tf.TensorSpec(
        shape=(
            None,
            WINDOW_LEN,
            N_FEATURES,
        ),
        dtype=tf.float32,
        name="motor_window",
    ),
)

onnx_model, _ = tf2onnx.convert.from_keras(
    final_model,
    input_signature=input_signature,
    opset=13,
    output_path=ONNX_PATH,
)

print("Saved ONNX:", ONNX_PATH)

print(
    "ONNX size KB:",
    round(
        os.path.getsize(ONNX_PATH) / 1024,
        2,
    ),
)



In [ ]:
# ==========================================================
# CELL 17 — full INT8 TFLite export
# ==========================================================
TFLITE_INT8_PATH = os.path.abspath(
    os.path.join(
        FINAL_DIR,
        "pure_mlp_ae_anomaly_classifier_int8.tflite",
    )
)


def representative_dataset():
    sample_count = min(
        500,
        len(X_all),
    )

    sample_indices = np.linspace(
        0,
        len(X_all) - 1,
        sample_count,
        dtype=np.int64,
    )

    for index in sample_indices:
        yield [
            X_all[
                index:index + 1
            ].astype(np.float32)
        ]


converter = tf.lite.TFLiteConverter.from_keras_model(
    final_model
)

converter.optimizations = [
    tf.lite.Optimize.DEFAULT
]

converter.representative_dataset = (
    representative_dataset
)

converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS_INT8
]

converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_int8_model = converter.convert()

with open(
    TFLITE_INT8_PATH,
    "wb",
) as file:
    file.write(
        tflite_int8_model
    )

tflite_size_kb = (
    os.path.getsize(
        TFLITE_INT8_PATH
    )
    / 1024
)

print("Saved INT8 TFLite:", TFLITE_INT8_PATH)
print("INT8 TFLite size KB:", round(tflite_size_kb, 2))

if tflite_size_kb <= 30.0:
    print(
        "PASS: INT8 model is within the 30 KB target."
    )
else:
    print(
        "WARNING: INT8 model exceeds the 30 KB target."
    )

# ==========================================================
# FLOAT32 TFLite export and download
# ==========================================================

TFLITE_FLOAT32_PATH = os.path.abspath(
    os.path.join(
        FINAL_DIR,
        "pure_mlp_ae_anomaly_classifier_float32.tflite",
    )
)

float32_converter = tf.lite.TFLiteConverter.from_keras_model(
    final_model
)

float32_converter.optimizations = []

tflite_float32_model = float32_converter.convert()

with open(
    TFLITE_FLOAT32_PATH,
    "wb",
) as file:
    file.write(
        tflite_float32_model
    )

float32_size_kb = (
    os.path.getsize(
        TFLITE_FLOAT32_PATH
    )
    / 1024
)

print(
    "Saved FLOAT32 TFLite:",
    TFLITE_FLOAT32_PATH,
)

print(
    "FLOAT32 TFLite size KB:",
    round(float32_size_kb, 2),
)

try:
    from google.colab import files

    files.download(
        TFLITE_FLOAT32_PATH
    )

except ImportError:
    print(
        "Not running in Google Colab."
    )
    print(
        "Download the file manually from:",
        TFLITE_FLOAT32_PATH,
    )

In [ ]:
# ==========================================================
# CELL 17.1 — recalibrate reconstruction thresholds against the
# ACTUAL deployed INT8 TFLite model, not the float32 Keras model.
# Overwrites the global threshold variables in place, so CELL 18
# and CELL 19 automatically embed the INT8-correct numbers.
# ==========================================================
int8_interpreter = tf.lite.Interpreter(model_path=TFLITE_INT8_PATH)
int8_interpreter.allocate_tensors()

int8_input_details = int8_interpreter.get_input_details()[0]

def identify_recon_output_detail(output_details, window_len, n_features):
    """
    TFLite/INT8 conversion frequently strips or renames the original
    Keras layer names (e.g. 'reconstruction_output' becomes something
    like 'PartitionedCall:0'), so name-based lookup is unreliable.
    Identify the reconstruction output by its shape instead: it's the
    only output whose per-sample element count equals window_len * n_features.
    """
    target_elements = window_len * n_features
    candidates = []

    for detail in output_details:
        shape = tuple(int(v) for v in detail["shape"])
        element_count = int(np.prod(shape[1:])) if len(shape) > 1 else int(shape[0])

        if element_count == target_elements:
            candidates.append(detail)

    if len(candidates) == 1:
        return candidates[0]

    if len(candidates) == 0:
        raise RuntimeError(
            "Could not identify the reconstruction output tensor by shape. "
            f"Expected {target_elements} elements per sample "
            f"({window_len} x {n_features}). Found output shapes: "
            f"{[tuple(int(v) for v in d['shape']) for d in output_details]}"
        )

    raise RuntimeError(
        "Multiple output tensors match the reconstruction shape; "
        "cannot disambiguate automatically. Found shapes: "
        f"{[tuple(int(v) for v in d['shape']) for d in candidates]}"
    )


int8_recon_output_details = identify_recon_output_detail(
    int8_interpreter.get_output_details(), WINDOW_LEN, N_FEATURES,
)

print(
    "Identified reconstruction output tensor:",
    int8_recon_output_details["name"],
    "| shape:", int8_recon_output_details["shape"],
)

input_scale, input_zero_point = int8_input_details["quantization"]
output_scale, output_zero_point = int8_recon_output_details["quantization"]


def int8_reconstruction_mse_batch(X_windows, X_normalized_reference):
    mse_values = np.empty(len(X_windows), dtype=np.float32)

    for i in range(len(X_windows)):
        x_raw = X_windows[i:i + 1]

        if input_scale and input_scale > 0:
            x_int8 = np.round(x_raw / input_scale + input_zero_point)
        else:
            x_int8 = x_raw

        x_int8 = np.clip(x_int8, -128, 127).astype(np.int8)

        int8_interpreter.set_tensor(int8_input_details["index"], x_int8)
        int8_interpreter.invoke()

        out_int8 = int8_interpreter.get_tensor(int8_recon_output_details["index"])

        if output_scale and output_scale > 0:
            out_float = (out_int8.astype(np.float32) - output_zero_point) * output_scale
        else:
            out_float = out_int8.astype(np.float32)

        mse_values[i] = np.mean(np.square(out_float - X_normalized_reference[i:i + 1]))

    return mse_values


print("Recomputing reconstruction MSE with the deployed INT8 model...")
int8_all_reconstruction_mse = int8_reconstruction_mse_batch(X_all, X_all_normalized)

pwm_feature_index = FEATURE_COLUMNS.index("PWM")
all_mean_pwm_for_calibration = np.mean(
    X_all[:, :, pwm_feature_index], axis=1,
).astype(np.float32)

int8_loaded_normal_mask = (reconstruction_reference_all == 1)
int8_loaded_normal_mse = int8_all_reconstruction_mse[int8_loaded_normal_mask]

RECON_THRESHOLD, RECON_THRESHOLD_CONFIDENCE = quantile_with_confidence(
    int8_loaded_normal_mse, RECON_QUANTILE,
)
GLOBAL_NORMAL_BASELINE = float(np.median(int8_loaded_normal_mse))

for state_label in ["motor_off", "turning_off", "turning_on"]:
    state_class_id = LABEL_TO_ID[state_label]
    state_errors = int8_all_reconstruction_mse[y_class_all == state_class_id]

    if len(state_errors) >= MIN_THRESHOLD_WINDOWS:
        state_threshold, state_confidence = quantile_with_confidence(state_errors, RECON_QUANTILE)
    else:
        state_threshold, state_confidence = RECON_THRESHOLD, 0.0
        print(f"WARNING (INT8): state {state_label} has only {len(state_errors)} windows.")

    STATE_RECON_THRESHOLDS[state_label] = state_threshold
    STATE_THRESHOLD_CONFIDENCE[state_label] = state_confidence

    print(
        f"[INT8] State {state_label} | windows={len(state_errors)} | "
        f"threshold={state_threshold:.8f} | confidence={state_confidence:.2f}"
    )

int8_normal_mean_pwm = all_mean_pwm_for_calibration[int8_loaded_normal_mask]

for bin_index, bin_name in enumerate(PWM_BIN_NAMES):
    lower_pwm = PWM_BIN_EDGES[bin_index]
    upper_pwm = PWM_BIN_EDGES[bin_index + 1]

    pwm_mask = (int8_normal_mean_pwm >= lower_pwm) & (int8_normal_mean_pwm < upper_pwm)
    pwm_errors = int8_loaded_normal_mse[pwm_mask]

    if len(pwm_errors) >= MIN_THRESHOLD_WINDOWS:
        pwm_threshold, pwm_confidence = quantile_with_confidence(pwm_errors, RECON_QUANTILE)
        pwm_baseline = float(np.median(pwm_errors))
    else:
        pwm_threshold, pwm_confidence = RECON_THRESHOLD, 0.0
        pwm_baseline = GLOBAL_NORMAL_BASELINE
        print(f"WARNING (INT8): PWM bin {bin_name} has only {len(pwm_errors)} windows.")

    NORMAL_PWM_RECON_THRESHOLDS[bin_name] = pwm_threshold
    LOCAL_NORMAL_BASELINES[bin_name] = pwm_baseline
    PWM_THRESHOLD_CONFIDENCE[bin_name] = pwm_confidence

    print(
        f"[INT8] Normal {bin_name} | windows={len(pwm_errors)} | "
        f"threshold={pwm_threshold:.8f} | baseline={pwm_baseline:.8f} | "
        f"confidence={pwm_confidence:.2f}"
    )

# Re-save thresholds.json / metadata.json with the INT8-correct values.
thresholds["global_reconstruction_fallback"] = RECON_THRESHOLD
thresholds["global_normal_baseline"] = GLOBAL_NORMAL_BASELINE
thresholds["state_reconstruction_thresholds"] = STATE_RECON_THRESHOLDS
thresholds["state_threshold_confidence"] = STATE_THRESHOLD_CONFIDENCE
thresholds["normal_pwm_reconstruction_thresholds"] = NORMAL_PWM_RECON_THRESHOLDS
thresholds["local_normal_baselines"] = LOCAL_NORMAL_BASELINES
thresholds["pwm_threshold_confidence"] = PWM_THRESHOLD_CONFIDENCE
thresholds["anomaly_probability_threshold"] = ANOMALY_PROB_THRESHOLD
thresholds["class_confidence_threshold"] = CLASS_CONF_THRESHOLD
thresholds["calibration_source"] = "int8_tflite"

metadata["thresholds"] = thresholds

with open(THRESHOLDS_PATH, "w", encoding="utf-8") as file:
    json.dump(thresholds, file, indent=2)

with open(METADATA_PATH, "w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=2)

print("\nRe-saved INT8-calibrated thresholds to:", THRESHOLDS_PATH)
print("Re-saved updated metadata to:", METADATA_PATH)

In [ ]:
# ==========================================================
# CELL 18 — export complete model as one C header
# ==========================================================
MODEL_HEADER_PATH = os.path.abspath(
    os.path.join(
        FINAL_DIR,
        "pure_mlp_ae_anomaly_classifier_model.h",
    )
)


def write_binary_model_header(
    binary_model_path,
    header_path,
    array_name,
):
    with open(
        binary_model_path,
        "rb",
    ) as file:
        model_bytes = file.read()

    include_guard = (
        array_name.upper()
        + "_H"
    )

    with open(
        header_path,
        "w",
        encoding="utf-8",
    ) as file:
        file.write(
            f"#ifndef {include_guard}\n"
        )

        file.write(
            f"#define {include_guard}\n\n"
        )

        file.write(
            "#include <stdint.h>\n"
        )

        file.write(
            "#include <stddef.h>\n\n"
        )

        file.write(
            "#if defined(__GNUC__)\n"
        )

        file.write(
            "#define MODEL_ALIGN "
            "__attribute__((aligned(16)))\n"
        )

        file.write(
            "#else\n"
        )

        file.write(
            "#define MODEL_ALIGN\n"
        )

        file.write(
            "#endif\n\n"
        )

        file.write(
            f"static const MODEL_ALIGN "
            f"uint8_t {array_name}"
            f"[{len(model_bytes)}] = {{\n"
        )

        for start in range(
            0,
            len(model_bytes),
            12,
        ):
            chunk = model_bytes[
                start:start + 12
            ]

            line = ", ".join(
                f"0x{byte:02X}"
                for byte in chunk
            )

            file.write(
                f"    {line},\n"
            )

        file.write(
            "};\n\n"
        )

        file.write(
            f"static const size_t "
            f"{array_name}_len = "
            f"{len(model_bytes)}u;\n\n"
        )

        file.write(
            f"#endif /* {include_guard} */\n"
        )

    return len(model_bytes)


header_model_size = write_binary_model_header(
    binary_model_path=TFLITE_INT8_PATH,
    header_path=MODEL_HEADER_PATH,
    array_name=(
        "pure_mlp_ae_anomaly_classifier_model"
    ),
)

print(
    "Saved complete model header:",
    MODEL_HEADER_PATH,
)

print(
    "Binary model size KB:",
    round(
        header_model_size / 1024,
        2,
    ),
)

In [ ]:
# ==========================================================
# CELL 19 — export firmware parameters header
# ==========================================================
PARAMS_HEADER_PATH = os.path.abspath(
    os.path.join(
        FINAL_DIR,
        "motor_model_params.h",
    )
)


def c_float(value):
    return f"{float(value):.9g}f"


normal_thresholds = np.asarray([
    NORMAL_PWM_RECON_THRESHOLDS[
        bin_name
    ]
    for bin_name in PWM_BIN_NAMES
], dtype=np.float32)

local_normal_baseline_values = np.asarray([
    LOCAL_NORMAL_BASELINES[
        bin_name
    ]
    for bin_name in PWM_BIN_NAMES
], dtype=np.float32)


with open(
    PARAMS_HEADER_PATH,
    "w",
    encoding="utf-8",
) as file:
    file.write("#ifndef MOTOR_MODEL_PARAMS_H\n")
    file.write("#define MOTOR_MODEL_PARAMS_H\n\n")
    file.write(
        f"#define MOTOR_MODEL_WINDOW_LEN {WINDOW_LEN}\n"
    )
    file.write(
        f"#define MOTOR_MODEL_N_FEATURES {N_FEATURES}\n"
    )
    file.write(
        "#define MOTOR_MODEL_INPUT_SIZE "
        f"{WINDOW_LEN * N_FEATURES}\n"
    )
    file.write(
        f"#define MOTOR_MODEL_N_CLASSES {N_CLASSES}\n"
    )
    file.write(
        f"#define MOTOR_MODEL_N_PWM_BINS {len(PWM_BIN_NAMES)}\n"
    )
    file.write(
        "#define MOTOR_MODEL_LOAD_PROXY_FEATURE_INDEX "
        f"{FEATURE_COLUMNS.index(LOAD_PROXY_FEATURE)}\n"
    )
    file.write(
        "#define MOTOR_MODEL_ANOMALY_THRESHOLD "
        f"{c_float(ANOMALY_PROB_THRESHOLD)}\n"
    )
    file.write(
        "#define MOTOR_MODEL_CLASS_CONF_THRESHOLD "
        f"{c_float(CLASS_CONF_THRESHOLD)}\n"
    )
    file.write(
        "#define MOTOR_MODEL_RECON_THRESHOLD "
        f"{c_float(RECON_THRESHOLD)}\n"
    )
    file.write(
        "#define MOTOR_MODEL_GLOBAL_NORMAL_BASELINE "
        f"{c_float(GLOBAL_NORMAL_BASELINE)}\n"
    )
    file.write(
        "#define MOTOR_MODEL_SEVERITY_MILD_MULTIPLIER "
        f"{c_float(SEVERITY_MILD_MULTIPLIER)}\n"
    )
    file.write(
        "#define MOTOR_MODEL_SEVERITY_MODERATE_MULTIPLIER "
        f"{c_float(SEVERITY_MODERATE_MULTIPLIER)}\n"
    )
    file.write(
        "#define MOTOR_MODEL_SEVERITY_SEVERE_MULTIPLIER "
        f"{c_float(SEVERITY_SEVERE_MULTIPLIER)}\n\n"
    )

    file.write(
        "static const float "
        f"MOTOR_MODEL_NORMALIZATION_MEAN[{N_FEATURES}] = {{\n    "
    )
    file.write(
        ", ".join(
            c_float(value)
            for value in final_mean
        )
    )
    file.write("\n};\n\n")

    file.write(
        "static const float "
        f"MOTOR_MODEL_NORMALIZATION_VARIANCE[{N_FEATURES}] = {{\n    "
    )
    file.write(
        ", ".join(
            c_float(value)
            for value in final_variance
        )
    )
    file.write("\n};\n\n")

    file.write(
        "static const float "
        f"MOTOR_MODEL_PWM_BIN_EDGES[{len(PWM_BIN_EDGES)}] = {{\n    "
    )
    file.write(
        ", ".join(
            c_float(value)
            for value in PWM_BIN_EDGES
        )
    )
    file.write("\n};\n\n")

    file.write(
        "static const float "
        f"MOTOR_MODEL_NORMAL_THRESHOLDS[{len(PWM_BIN_NAMES)}] = {{\n    "
    )
    file.write(
        ", ".join(
            c_float(value)
            for value in normal_thresholds
        )
    )
    file.write("\n};\n\n")

    file.write(
        "static const float "
        f"MOTOR_MODEL_LOCAL_NORMAL_BASELINES[{len(PWM_BIN_NAMES)}] = {{\n    "
    )
    file.write(
        ", ".join(
            c_float(value)
            for value in local_normal_baseline_values
        )
    )
    file.write("\n};\n\n")

    file.write(
        "#define MOTOR_MODEL_MOTOR_OFF_RECON_THRESHOLD "
        f"{c_float(STATE_RECON_THRESHOLDS['motor_off'])}\n"
    )
    file.write(
        "#define MOTOR_MODEL_TURNING_OFF_RECON_THRESHOLD "
        f"{c_float(STATE_RECON_THRESHOLDS['turning_off'])}\n"
    )
    file.write(
        "#define MOTOR_MODEL_TURNING_ON_RECON_THRESHOLD "
        f"{c_float(STATE_RECON_THRESHOLDS['turning_on'])}\n\n"
    )

    file.write(
        "static const char *const "
        f"MOTOR_MODEL_LABELS[{N_CLASSES}] = {{\n"
    )
    for label in FINAL_LABELS:
        file.write(f'    "{label}",\n')
    file.write("};\n\n")
    file.write("#endif /* MOTOR_MODEL_PARAMS_H */\n")

print("Saved firmware parameters header:", PARAMS_HEADER_PATH)

In [ ]:
# ==========================================================
# CELL 20 — verify TFLite model interfaces
# ==========================================================
interpreter = tf.lite.Interpreter(
    model_path=TFLITE_INT8_PATH
)

interpreter.allocate_tensors()

print("TFLite input details:")

for detail in interpreter.get_input_details():
    print(detail)

print("TFLite output details:")

for detail in interpreter.get_output_details():
    print(detail)

In [ ]:

# ==========================================================
# CELL 21 — Manual classification and anomaly test
# Correct sigmoid conversion for raw anomaly logits
# ==========================================================

def manual_classification_prediction(test_file):

    model = tf.keras.models.load_model(
        KERAS_PATH,
        compile=False,
    )

    test_df = read_one(test_file)

    X_test, starts = make_windows_from_array(
        test_df.to_numpy(dtype=np.float32)
    )

    if X_test is None or len(X_test) == 0:
        raise RuntimeError(
            "Selected file has insufficient rows "
            "for one 30-row window."
        )

    X_test_normalized = normalize_windows_numpy(
        X_test,
        final_mean,
        final_variance,
    )

    prediction_dict = model.predict(
        X_test,
        batch_size=BATCH_SIZE,
        verbose=0,
    )

    reconstructed = prediction_dict[
        "reconstruction_output"
    ]

    class_probabilities = prediction_dict[
        "class_output"
    ]

    # Convert raw anomaly logits to probabilities ONCE
    anomaly_logits = prediction_dict[
        "anomaly_output"
    ].reshape(-1)

    anomaly_probabilities = tf.math.sigmoid(
        anomaly_logits
    ).numpy().reshape(-1)

    reconstruction_mse = np.mean(
        np.square(
            reconstructed - X_test_normalized
        ),
        axis=(1, 2),
    )

    pwm_feature_index = FEATURE_COLUMNS.index("PWM")
    irms_feature_index = FEATURE_COLUMNS.index(
        LOAD_PROXY_FEATURE
    )

    window_mean_pwm = np.mean(
        X_test[:, :, pwm_feature_index],
        axis=1,
    ).astype(np.float32)

    window_mean_irms = np.mean(
        X_test[:, :, irms_feature_index],
        axis=1,
    ).astype(np.float32)

    preliminary_class_predictions = np.argmax(
        class_probabilities,
        axis=1,
    ).astype(np.int64)

    # Reconstruction thresholds
    window_reconstruction_thresholds = np.asarray([
        get_conditional_reconstruction_threshold(
            class_id, pwm, irms
        )
        for class_id, pwm, irms in zip(
            preliminary_class_predictions,
            window_mean_pwm,
            window_mean_irms,
        )
    ], dtype=np.float32)

    # Existing decision fusion
    (
        class_predictions,
        class_confidence,
        binary_anomaly_flags,
        reconstruction_anomaly_flags,
        anomaly_flags,
        final_decisions,
    ) = fuse_decisions(
        class_probabilities,
        anomaly_probabilities,
        reconstruction_mse,
        window_reconstruction_thresholds,
    )

    # Per-window results
    result = pd.DataFrame({
        "window": np.arange(len(X_test)),
        "start_row": starts,
        "time_sec": (
            starts + WINDOW_LEN / 2.0
        ) * ROW_PERIOD_SEC,

        "mean_pwm": window_mean_pwm,
        "mean_irms": window_mean_irms,

        "pred_class_id": class_predictions,
        "pred_label": [
            FINAL_LABELS[int(index)]
            for index in class_predictions
        ],

        "class_confidence": class_confidence,

        "anomaly_logit": anomaly_logits,
        "anomaly_probability": anomaly_probabilities,

        "binary_anomaly_flag":
            binary_anomaly_flags.astype(np.int64),

        "reconstruction_mse": reconstruction_mse,
        "reconstruction_threshold":
            window_reconstruction_thresholds,

        "reconstruction_anomaly_flag":
            reconstruction_anomaly_flags.astype(np.int64),

        "anomaly_flag":
            anomaly_flags.astype(np.int64),

        "final_decision": final_decisions,
    })

    for class_id, label in enumerate(FINAL_LABELS):
        result[
            f"probability_{label}"
        ] = class_probabilities[:, class_id]

    majority_class_id = int(
        result["pred_class_id"].value_counts().idxmax()
    )

    majority_label = FINAL_LABELS[
        majority_class_id
    ]

    majority_final_decision = (
        result["final_decision"].value_counts().idxmax()
    )

    stem = Path(test_file).stem

    output_csv = os.path.join(
        MANUAL_TEST_DIR,
        f"manual_test_{stem}.csv",
    )

    result.to_csv(output_csv, index=False)

    print("\n" + "=" * 70)
    print("MANUAL PURE MLP ANN 3-HEAD TEST")
    print("=" * 70)

    print("File:", test_file)
    print("Total windows:", len(result))
    print("Majority class:", majority_label)
    print(
        "Majority final decision:",
        majority_final_decision,
    )
    print(
        "Mean class confidence:",
        float(np.mean(class_confidence)),
    )
    print(
        "Mean anomaly probability:",
        float(np.mean(anomaly_probabilities)),
    )
    print(
        "Mean reconstruction MSE:",
        float(np.mean(reconstruction_mse)),
    )
    print(
        "Anomalous windows:",
        int(np.sum(anomaly_flags)),
        "/", len(result),
    )
    print("Saved CSV:", output_csv)

    print("\nFinal decisions:")
    print(result["final_decision"].value_counts())

    # ======================================================
    # CHART 1 — Multi-class classification head
    # ======================================================
    plt.figure(figsize=(14, 7))

    for class_id, label in enumerate(FINAL_LABELS):
        plt.plot(
            result["time_sec"],
            class_probabilities[:, class_id],
            label=label,
            linewidth=1.5,
        )

    plt.axhline(
        y=CLASS_CONF_THRESHOLD,
        linestyle="--",
        label=(
            f"Confidence threshold "
            f"({CLASS_CONF_THRESHOLD:.2f})"
        ),
    )

    plt.xlabel("Time [s]")
    plt.ylabel("Class probability")
    plt.ylim(0.0, 1.05)
    plt.title(
        f"Classification head probabilities — {stem}"
    )
    plt.legend(loc="upper right", ncol=2)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    class_head_path = os.path.join(
        MANUAL_TEST_DIR,
        f"manual_test_{stem}_class_head.png",
    )

    plt.savefig(class_head_path, dpi=180)
    plt.show()
    plt.close()

    print("Saved classification chart:", class_head_path)

    # ======================================================
    # CHART 2 — Binary anomaly head
    # ======================================================
    plt.figure(figsize=(14, 5))

    plt.plot(
        result["time_sec"],
        anomaly_probabilities,
        label="Anomaly probability",
        linewidth=2,
    )

    plt.axhline(
        y=ANOMALY_PROB_THRESHOLD,
        linestyle="--",
        label=(
            f"Anomaly threshold "
            f"({ANOMALY_PROB_THRESHOLD:.3f})"
        ),
    )

    plt.fill_between(
        result["time_sec"],
        0,
        1,
        where=(
            anomaly_probabilities
            >= ANOMALY_PROB_THRESHOLD
        ),
        alpha=0.15,
        label="Anomaly detected",
    )

    plt.xlabel("Time [s]")
    plt.ylabel("Anomaly probability")
    plt.ylim(0.0, 1.05)
    plt.title(f"Binary anomaly head — {stem}")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    anomaly_head_path = os.path.join(
        MANUAL_TEST_DIR,
        f"manual_test_{stem}_anomaly_head.png",
    )

    plt.savefig(anomaly_head_path, dpi=180)
    plt.show()
    plt.close()

    print("Saved anomaly chart:", anomaly_head_path)

    # ======================================================
    # CHART 3 — Autoencoder reconstruction head
    # ======================================================
    plt.figure(figsize=(14, 5))

    plt.plot(
        result["time_sec"],
        reconstruction_mse,
        label="Reconstruction MSE",
        linewidth=2,
    )

    plt.plot(
        result["time_sec"],
        window_reconstruction_thresholds,
        linestyle="--",
        linewidth=2,
        label="State/PWM-conditioned threshold",
    )

    plt.fill_between(
        result["time_sec"],
        0,
        reconstruction_mse,
        where=(
            reconstruction_mse
            > window_reconstruction_thresholds
        ),
        alpha=0.15,
        label="Reconstruction anomaly",
    )

    plt.xlabel("Time [s]")
    plt.ylabel("Reconstruction MSE")
    plt.title(
        f"Autoencoder reconstruction head — {stem}"
    )
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    reconstruction_head_path = os.path.join(
        MANUAL_TEST_DIR,
        f"manual_test_{stem}_reconstruction_head.png",
    )

    plt.savefig(reconstruction_head_path, dpi=180)
    plt.show()
    plt.close()

    print(
        "Saved reconstruction chart:",
        reconstruction_head_path,
    )

    # ======================================================
    # CHART 4 — Predicted class timeline
    # ======================================================
    plt.figure(figsize=(14, 5))

    plt.step(
        result["time_sec"],
        result["pred_class_id"],
        where="mid",
        linewidth=2,
    )

    plt.yticks(
        np.arange(N_CLASSES),
        FINAL_LABELS,
    )

    plt.xlabel("Time [s]")
    plt.ylabel("Predicted class")
    plt.title(
        f"Classification result timeline — {stem}"
    )
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    timeline_path = os.path.join(
        MANUAL_TEST_DIR,
        f"manual_test_{stem}_class_timeline.png",
    )

    plt.savefig(timeline_path, dpi=180)
    plt.show()
    plt.close()

    print("Saved class timeline:", timeline_path)

    return result


In [ ]:

# ==========================================================
# CELL 22 — Select manual test file
# ==========================================================

import tkinter as tk
from tkinter import filedialog

root = tk.Tk()
root.withdraw()
root.attributes("-topmost", True)

TEST_FILE = filedialog.askopenfilename(
    title="Select motor CSV or Excel file",
    filetypes=[
        ("Excel files", "*.xlsx"),
        ("CSV files", "*.csv"),
        ("All files", "*.*"),
    ],
)

root.destroy()

if TEST_FILE:
    manual_result = manual_classification_prediction(
        TEST_FILE
    )
else:
    print("No file selected.")


In [ ]:
verify = manual_result.copy()
verify["recon_ratio"] = verify["reconstruction_mse"] / verify["reconstruction_threshold"]

# only look at windows where the model itself said "normal" —
# these are the ones that should have ratio comfortably under 1.0
healthy_windows = verify[verify["pred_label"] == "normal"]

def bin_name_for_pwm(pwm):
    idx = get_pwm_bin_index(pwm)
    return PWM_BIN_NAMES[idx]

healthy_windows = healthy_windows.copy()
healthy_windows["pwm_bin"] = healthy_windows["mean_pwm"].apply(bin_name_for_pwm)

summary = healthy_windows.groupby("pwm_bin")["recon_ratio"].agg(
    n="count",
    mean="mean",
    p90=lambda x: x.quantile(0.90),
    max="max",
).reindex(PWM_BIN_NAMES)

print(summary.to_string())
print()
print("Bins where healthy windows exceed 1.0 anywhere (still broken):")
print(summary[summary["max"] >= 1.0])

In [ ]:
# # ==========================================================
# # CELL 23 — STANDALONE DANN / MLP 3-HEAD MANUAL TEST/ int8 model dann 
# # Run ONLY this cell after restarting the kernel.
# #
# # Loads:
# #   - Saved INT8 TFLite model
# #   - metadata.json
# #   - thresholds.json
# #
# # Outputs:
# #   - Detailed CSV
# #   - 5 charts
# #   - Prediction and severity summaries
# #
# # No training or recalibration.
# # ==========================================================

# import json
# from pathlib import Path
# import tkinter as tk
# from tkinter import filedialog

# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# import tensorflow as tf


# # ==========================================================
# # 1. FIND SAVED DANN MODEL
# # ==========================================================

# OUTPUT_ROOT = Path.cwd() / "mlp_ae_ann_3head_output"
# MODEL_DIR = OUTPUT_ROOT / "final_model"

# MODEL_PATH = MODEL_DIR / "pure_mlp_ae_anomaly_classifier_int8.tflite"
# METADATA_PATH = MODEL_DIR / "metadata.json"
# THRESHOLDS_PATH = MODEL_DIR / "thresholds.json"


# def select_file(title, filetypes, initialdir=None):
#     root = tk.Tk()
#     root.withdraw()
#     root.attributes("-topmost", True)

#     path = filedialog.askopenfilename(
#         title=title,
#         initialdir=str(initialdir or Path.cwd()),
#         filetypes=filetypes
#     )

#     root.destroy()
#     return Path(path) if path else None


# if not MODEL_PATH.is_file():
#     print("Saved DANN INT8 model not found in default directory.")

#     MODEL_PATH = select_file(
#         "Select trained DANN INT8 TFLite model",
#         [("TFLite model", "*.tflite")]
#     )

#     if MODEL_PATH is None:
#         raise RuntimeError("No DANN model selected.")

#     MODEL_DIR = MODEL_PATH.parent
#     OUTPUT_ROOT = MODEL_DIR.parent
#     METADATA_PATH = MODEL_DIR / "metadata.json"
#     THRESHOLDS_PATH = MODEL_DIR / "thresholds.json"


# if not METADATA_PATH.is_file():
#     raise FileNotFoundError(METADATA_PATH)

# with open(METADATA_PATH, encoding="utf-8") as f:
#     meta = json.load(f)

# if THRESHOLDS_PATH.is_file():
#     with open(THRESHOLDS_PATH, encoding="utf-8") as f:
#         thresholds = json.load(f)
# else:
#     thresholds = meta["thresholds"]

# if thresholds.get("calibration_source") != "int8_tflite":
#     raise RuntimeError(
#         "The saved thresholds are not identified as INT8-calibrated. "
#         "Run the notebook's Cell 17.1 once before using this cell."
#     )


# # ==========================================================
# # 2. LOAD SAVED CONFIGURATION
# # ==========================================================

# FEATURES = meta["feature_columns"]
# LABELS = meta["labels"]
# SAFE_LABELS = meta["safe_labels"]
# FAULT_LABELS = meta["fault_labels"]

# WIN, NF = map(int, meta["input_shape"])
# STEP = int(meta["window_step"])
# ROW_SEC = float(meta["row_period_sec"])
# SCALE = float(meta["scale_divisor"])

# MEAN = np.asarray(
#     meta["normalization_mean"], dtype=np.float32
# )

# VAR = np.asarray(
#     meta["normalization_variance"], dtype=np.float32
# )

# ANOM_THR = float(
#     thresholds["anomaly_probability_threshold"]
# )

# CLASS_THR = float(
#     thresholds["class_confidence_threshold"]
# )

# GLOBAL_RECON_THR = float(
#     thresholds["global_reconstruction_fallback"]
# )

# GLOBAL_BASELINE = float(
#     thresholds["global_normal_baseline"]
# )

# STATE_THRESHOLDS = thresholds[
#     "state_reconstruction_thresholds"
# ]

# PWM_EDGES = np.asarray(
#     thresholds["pwm_bin_edges"], dtype=np.float32
# )

# PWM_NAMES = thresholds["pwm_bin_names"]

# NORMAL_PWM_THRESHOLDS = thresholds[
#     "normal_pwm_reconstruction_thresholds"
# ]

# LOCAL_BASELINES = thresholds["local_normal_baselines"]

# # Original DANN Cell 2 severity settings
# MILD = 2.0
# MODERATE = 4.0
# SEVERE = 8.0

# BATCH_SIZE = 128

# print("DANN model:", MODEL_PATH)
# print("Model input:", WIN, "x", NF)
# print("Anomaly threshold:", ANOM_THR)
# print("Class confidence threshold:", CLASS_THR)


# # ==========================================================
# # 3. SELECT MANUAL TEST FILE
# # ==========================================================

# TEST_FILE = select_file(
#     "Select motor CSV or Excel file for DANN manual testing",
#     [
#         ("Motor data files", "*.xlsx *.csv"),
#         ("Excel files", "*.xlsx"),
#         ("CSV files", "*.csv"),
#         ("All files", "*.*")
#     ]
# )

# if TEST_FILE is None:
#     raise RuntimeError("Manual test cancelled.")

# RESULT_DIR = OUTPUT_ROOT / "manual_test_results"
# RESULT_DIR.mkdir(parents=True, exist_ok=True)

# stem = TEST_FILE.stem

# print("\nSelected file:", TEST_FILE)
# print("Results directory:", RESULT_DIR)


# # ==========================================================
# # 4. READ AND PREPROCESS DATA — SAME AS DANN CELL 4
# # ==========================================================

# if TEST_FILE.suffix.lower() == ".xlsx":
#     df = pd.read_excel(TEST_FILE)

# elif TEST_FILE.suffix.lower() == ".csv":
#     df = pd.read_csv(TEST_FILE)

# else:
#     raise ValueError("Only CSV and XLSX files supported.")

# missing = [c for c in FEATURES if c not in df.columns]

# if missing:
#     raise ValueError(f"Missing required feature columns: {missing}")

# data = df[FEATURES].apply(
#     pd.to_numeric, errors="coerce"
# )

# data = data / SCALE

# # Same PWM clipping as original DANN preprocessing.
# data["PWM"] = data["PWM"].clip(upper=1.0)

# data = data.replace(
#     [np.inf, -np.inf], np.nan
# ).dropna().reset_index(drop=True)

# X = data.to_numpy(dtype=np.float32)

# if len(X) < WIN:
#     raise ValueError(
#         f"Only {len(X)} valid rows. Minimum {WIN} required."
#     )

# starts = np.arange(
#     0, len(X) - WIN + 1, STEP, dtype=np.int64
# )

# windows = np.stack([
#     X[s:s + WIN] for s in starts
# ]).astype(np.float32)

# print("Valid rows:", len(X))
# print("Total windows:", len(windows))


# # ==========================================================
# # 5. LOAD DANN INT8 TFLITE MODEL
# # ==========================================================

# interpreter = tf.lite.Interpreter(model_path=str(MODEL_PATH))
# interpreter.allocate_tensors()

# input_details = interpreter.get_input_details()
# output_details = interpreter.get_output_details()

# if len(input_details) != 1:
#     raise RuntimeError("Expected one model input.")

# input_detail = input_details[0]

# if input_detail["dtype"] != np.int8:
#     raise TypeError(
#         "Selected model is not INT8. Choose the *_int8.tflite file."
#     )


# def quantize_input(values, detail):
#     scale, zero_point = detail["quantization"]

#     if scale <= 0:
#         raise RuntimeError("Invalid INT8 input quantization scale.")

#     q = np.round(values / scale + zero_point)
#     q = np.clip(q, -128, 127)

#     return q.astype(np.int8)


# def dequantize_output(values, detail):
#     if detail["dtype"] == np.float32:
#         return values.astype(np.float32)

#     scale, zero_point = detail["quantization"]

#     if scale <= 0:
#         raise RuntimeError(
#             f"Invalid quantization scale: {detail['name']}"
#         )

#     return (
#         values.astype(np.float32) - float(zero_point)
#     ) * float(scale)


# def output_role(detail):
#     shape = tuple(int(v) for v in detail["shape"])
#     elements = int(np.prod(shape[1:]))

#     if elements == WIN * NF:
#         return "reconstruction_output"

#     if elements == len(LABELS):
#         return "class_output"

#     if elements == 1:
#         return "anomaly_output"

#     raise RuntimeError(
#         f"Unknown output shape: {detail['name']} {shape}"
#     )


# # ==========================================================
# # 6. RUN DANN INT8 INFERENCE
# # ==========================================================

# outputs = {
#     "reconstruction_output": [],
#     "anomaly_output": [],
#     "class_output": []
# }

# for start in range(0, len(windows), BATCH_SIZE):

#     batch = windows[start:start + BATCH_SIZE]

#     desired_shape = [len(batch), WIN, NF]

#     if list(input_detail["shape"]) != desired_shape:
#         interpreter.resize_tensor_input(
#             input_detail["index"],
#             desired_shape,
#             strict=False
#         )

#         interpreter.allocate_tensors()
#         input_detail = interpreter.get_input_details()[0]

#     quantized = quantize_input(batch, input_detail)

#     interpreter.set_tensor(
#         input_detail["index"],
#         quantized
#     )

#     interpreter.invoke()

#     for detail in interpreter.get_output_details():

#         role = output_role(detail)

#         raw = interpreter.get_tensor(detail["index"])
#         values = dequantize_output(raw, detail)

#         if role == "reconstruction_output":
#             values = values.reshape(-1, WIN, NF)

#         elif role == "class_output":
#             values = values.reshape(-1, len(LABELS))

#         else:
#             values = values.reshape(-1)

#         outputs[role].append(values.astype(np.float32))


# predictions = {
#     role: np.concatenate(values, axis=0)
#     for role, values in outputs.items()
# }

# reconstructed = predictions["reconstruction_output"]
# class_probability = predictions["class_output"]

# # IMPORTANT: DANN anomaly output is a RAW LOGIT.
# anomaly_logit = predictions["anomaly_output"]

# # Stable sigmoid — applied exactly once.
# anomaly_probability = np.exp(
#     -np.logaddexp(0.0, -anomaly_logit.astype(np.float64))
# ).astype(np.float32)


# # ==========================================================
# # 7. CALCULATE RECONSTRUCTION MSE
# # ==========================================================

# normalized = (
#     windows - MEAN.reshape(1, 1, NF)
# ) / np.sqrt(
#     VAR.reshape(1, 1, NF) + 1e-7
# )

# reconstruction_mse = np.mean(
#     np.square(reconstructed - normalized),
#     axis=(1, 2)
# ).astype(np.float32)

# PWM_INDEX = FEATURES.index("PWM")
# IRMS_INDEX = FEATURES.index(
#     thresholds.get("load_proxy_feature", "Irms")
# )

# mean_pwm = windows[:, :, PWM_INDEX].mean(axis=1)
# mean_irms = windows[:, :, IRMS_INDEX].mean(axis=1)

# class_id = np.argmax(
#     class_probability, axis=1
# ).astype(np.int64)

# class_confidence = np.max(
#     class_probability, axis=1
# )


# # ==========================================================
# # 8. DANN STATE/PWM RECONSTRUCTION THRESHOLDS
# # ==========================================================

# def pwm_bin_index(pwm):
#     return int(np.clip(
#         np.searchsorted(
#             PWM_EDGES, pwm, side="right"
#         ) - 1,
#         0,
#         len(PWM_NAMES) - 1
#     ))


# def reconstruction_threshold(cid, pwm, irms):
#     label = LABELS[int(cid)]

#     if label in ["motor_off", "turning_off", "turning_on"]:
#         return float(
#             STATE_THRESHOLDS.get(label, GLOBAL_RECON_THR)
#         )

#     name = PWM_NAMES[pwm_bin_index(pwm)]

#     return float(
#         NORMAL_PWM_THRESHOLDS.get(
#             name, GLOBAL_RECON_THR
#         )
#     )


# recon_thresholds = np.asarray([
#     reconstruction_threshold(cid, pwm, irms)
#     for cid, pwm, irms in zip(
#         class_id, mean_pwm, mean_irms
#     )
# ], dtype=np.float32)


# # ==========================================================
# # 9. ORIGINAL DANN THREE-HEAD DECISION FUSION
# # ==========================================================

# binary_flags = anomaly_probability >= ANOM_THR
# reconstruction_flags = reconstruction_mse > recon_thresholds

# known_fault_flags = (
#     np.isin(
#         [LABELS[int(i)] for i in class_id],
#         FAULT_LABELS
#     )
#     & (class_confidence >= CLASS_THR)
# )

# anomaly_flags = (
#     binary_flags
#     | reconstruction_flags
#     | known_fault_flags
# )

# final_decisions = []

# for i in range(len(class_id)):

#     label = LABELS[int(class_id[i])]

#     if known_fault_flags[i]:
#         decision = label

#     elif binary_flags[i] or reconstruction_flags[i]:
#         decision = "unknown_fault"

#     elif (
#         label in SAFE_LABELS
#         and class_confidence[i] >= CLASS_THR
#     ):
#         decision = label

#     else:
#         decision = "uncertain_safe"

#     final_decisions.append(decision)


# # ==========================================================
# # 10. DANN RECONSTRUCTION SEVERITY
# # ==========================================================

# local_baselines = np.asarray([
#     float(
#         LOCAL_BASELINES.get(
#             PWM_NAMES[pwm_bin_index(pwm)],
#             GLOBAL_BASELINE
#         )
#     )
#     for pwm in mean_pwm
# ], dtype=np.float32)

# safe_baselines = np.where(
#     local_baselines > 0,
#     local_baselines,
#     1.0
# )

# severity_ratio = reconstruction_mse / safe_baselines

# severity_labels = []
# recommended_actions = []

# for ratio in severity_ratio:

#     if ratio < MILD:
#         severity_labels.append("normal")
#         recommended_actions.append("No action needed.")

#     elif ratio < MODERATE:
#         severity_labels.append("mild")
#         recommended_actions.append(
#             "Log and monitor; no immediate action."
#         )

#     elif ratio < SEVERE:
#         severity_labels.append("moderate")
#         recommended_actions.append(
#             "Schedule inspection soon."
#         )

#     else:
#         severity_labels.append("severe")
#         recommended_actions.append(
#             "Stop motor / inspect immediately."
#         )


# # ==========================================================
# # 11. EARLY WARNING — ADDITIONAL OUTPUT
# # ==========================================================

# early_candidates = (
#     (class_id == LABELS.index("normal"))
#     & reconstruction_flags
# )

# early_streak = np.zeros(
#     len(early_candidates), dtype=np.int32
# )

# for i, candidate in enumerate(early_candidates):
#     if candidate:
#         early_streak[i] = (
#             early_streak[i - 1] + 1 if i > 0 else 1
#         )

# early_flags = early_streak >= 3


# # ==========================================================
# # 12. CREATE DETAILED RESULT CSV
# # ==========================================================

# time_sec = (starts + WIN / 2.0) * ROW_SEC

# result = pd.DataFrame({

#     "window": np.arange(len(windows)),
#     "start_row": starts,
#     "time_sec": time_sec,

#     "mean_pwm": mean_pwm,
#     "mean_irms": mean_irms,

#     "pred_class_id": class_id,
#     "pred_label": [
#         LABELS[int(i)] for i in class_id
#     ],

#     "class_confidence": class_confidence,

#     "anomaly_logit": anomaly_logit,
#     "anomaly_probability": anomaly_probability,

#     "binary_anomaly_flag": binary_flags.astype(int),

#     "reconstruction_mse": reconstruction_mse,
#     "reconstruction_threshold": recon_thresholds,

#     "local_baseline": local_baselines,
#     "severity_ratio": severity_ratio,
#     "severity_label": severity_labels,
#     "recommended_action": recommended_actions,

#     "reconstruction_anomaly_flag": reconstruction_flags.astype(int),
#     "anomaly_flag": anomaly_flags.astype(int),
#     "final_decision": final_decisions,

#     "early_warning_candidate": early_candidates.astype(int),
#     "early_warning_streak": early_streak,
#     "early_warning_flag": early_flags.astype(int)
# })

# for i, label in enumerate(LABELS):
#     result[f"probability_{label}"] = (
#         class_probability[:, i]
#     )

# CSV_PATH = RESULT_DIR / f"manual_test_{stem}.csv"

# result.to_csv(CSV_PATH, index=False)


# # ==========================================================
# # 13. DISPLAY DANN TEST SUMMARY
# # ==========================================================

# print("\n" + "=" * 70)
# print("MANUAL DANN / PURE MLP ANN 3-HEAD TEST — INT8")
# print("=" * 70)

# print("File:", TEST_FILE.name)
# print("Total windows:", len(result))

# print(
#     "Majority class:",
#     result["pred_label"].value_counts().idxmax()
# )

# print(
#     "Majority final decision:",
#     result["final_decision"].value_counts().idxmax()
# )

# print(
#     "Majority severity:",
#     result["severity_label"].value_counts().idxmax()
# )

# print(
#     "Mean class confidence:",
#     float(np.mean(class_confidence))
# )

# print(
#     "Mean anomaly probability:",
#     float(np.mean(anomaly_probability))
# )

# print(
#     "Mean reconstruction MSE:",
#     float(np.mean(reconstruction_mse))
# )

# print(
#     "Anomalous windows:",
#     int(anomaly_flags.sum()), "/", len(result)
# )

# print(
#     "Early-warning windows:",
#     int(early_flags.sum()), "/", len(result)
# )

# if early_flags.any():
#     first_time = float(time_sec[np.flatnonzero(early_flags)[0]])
#     print("First EARLY_WARNING:", first_time, "s")
# else:
#     print("No EARLY_WARNING generated.")

# print("\nFinal decisions:")
# print(result["final_decision"].value_counts())

# print("\nSeverity tiers:")
# print(result["severity_label"].value_counts())

# print("\nSaved CSV:", CSV_PATH)


# # ==========================================================
# # 14. SAVE FIVE MANUAL-TEST CHARTS
# # ==========================================================

# def save_chart(suffix):
#     path = RESULT_DIR / f"manual_test_{stem}_{suffix}.png"
#     plt.tight_layout()
#     plt.savefig(path, dpi=180)
#     plt.show()
#     plt.close()
#     print("Saved:", path)


# # CHART 1 — DANN classification probabilities

# plt.figure(figsize=(14, 7))

# for i, label in enumerate(LABELS):
#     plt.plot(
#         time_sec,
#         class_probability[:, i],
#         label=label,
#         linewidth=1.5
#     )

# plt.axhline(
#     CLASS_THR,
#     linestyle="--",
#     label=f"Confidence threshold ({CLASS_THR:.2f})"
# )

# plt.xlabel("Time [s]")
# plt.ylabel("Class probability")
# plt.ylim(0, 1.05)
# plt.title(f"DANN classification head probabilities — {stem}")
# plt.legend(loc="upper right", ncol=2)
# plt.grid(True, alpha=0.3)

# save_chart("class_head")


# # CHART 2 — DANN binary anomaly probability

# plt.figure(figsize=(14, 5))

# plt.plot(
#     time_sec,
#     anomaly_probability,
#     label="Anomaly probability",
#     linewidth=2
# )

# plt.axhline(
#     ANOM_THR,
#     linestyle="--",
#     label=f"Anomaly threshold ({ANOM_THR:.3f})"
# )

# plt.fill_between(
#     time_sec,
#     0,
#     1,
#     where=binary_flags,
#     alpha=0.15,
#     label="Anomaly detected"
# )

# plt.xlabel("Time [s]")
# plt.ylabel("Anomaly probability")
# plt.ylim(0, 1.05)
# plt.title(f"DANN binary anomaly head — {stem}")
# plt.legend()
# plt.grid(True, alpha=0.3)

# save_chart("anomaly_head")


# # CHART 3 — DANN reconstruction MSE

# plt.figure(figsize=(14, 5))

# plt.plot(
#     time_sec,
#     reconstruction_mse,
#     label="Reconstruction MSE",
#     linewidth=2
# )

# plt.plot(
#     time_sec,
#     recon_thresholds,
#     linestyle="--",
#     linewidth=2,
#     label="State/PWM-conditioned threshold"
# )

# plt.fill_between(
#     time_sec,
#     0,
#     reconstruction_mse,
#     where=reconstruction_flags,
#     alpha=0.15,
#     label="Reconstruction anomaly"
# )

# plt.xlabel("Time [s]")
# plt.ylabel("Reconstruction MSE")
# plt.title(f"DANN autoencoder reconstruction — {stem}")
# plt.legend()
# plt.grid(True, alpha=0.3)

# save_chart("reconstruction_head")


# # CHART 4 — DANN predicted class timeline

# plt.figure(figsize=(14, 5))

# plt.step(
#     time_sec,
#     class_id,
#     where="mid",
#     linewidth=2
# )

# plt.yticks(np.arange(len(LABELS)), LABELS)
# plt.xlabel("Time [s]")
# plt.ylabel("Predicted class")
# plt.title(f"DANN classification result timeline — {stem}")
# plt.grid(True, alpha=0.3)

# save_chart("class_timeline")


# # CHART 5 — DANN reconstruction severity

# plt.figure(figsize=(14, 5))

# plt.plot(
#     time_sec,
#     severity_ratio,
#     linewidth=2,
#     color="darkorange",
#     label="Severity ratio (MSE / local baseline)"
# )

# for level, color in [
#     (MILD, "gold"),
#     (MODERATE, "orange"),
#     (SEVERE, "red")
# ]:
#     plt.axhline(
#         level,
#         linestyle="--",
#         color=color,
#         alpha=0.7
#     )

# plt.xlabel("Time [s]")
# plt.ylabel("Severity ratio")
# plt.title(f"DANN reconstruction severity — {stem}")
# plt.legend()
# plt.grid(True, alpha=0.3)

# save_chart("severity")


# # ==========================================================
# # 15. COMPLETE
# # ==========================================================

# manual_result = result

# print("\n" + "=" * 70)
# print("DANN MANUAL TEST COMPLETED SUCCESSFULLY")
# print("=" * 70)

# print("Model:", MODEL_PATH.name)
# print("Test file:", TEST_FILE.name)
# print("CSV:", CSV_PATH)
# print("Five charts saved in:", RESULT_DIR)
# print("No training or recalibration performed.")

In [ ]:
# ==========================================================
# CELL 23 — STANDALONE DANN / MLP 3-HEAD MANUAL TEST/ float32 model dann 
# Run ONLY this cell after restarting the kernel.
#
# Loads:
#   - Saved float32 TFLite model
#   - metadata.json
#   - thresholds.json
#
# Outputs:
#   - Detailed CSV
#   - 5 charts
#   - Prediction and severity summaries
#
# No training or recalibration.
# ==========================================================

import json
from pathlib import Path
import tkinter as tk
from tkinter import filedialog

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf


# ==========================================================
# 1. FIND SAVED DANN MODEL
# ==========================================================

OUTPUT_ROOT = Path.cwd() / "mlp_ae_ann_3head_output"
MODEL_DIR = OUTPUT_ROOT / "final_model"

MODEL_PATH = MODEL_DIR / "pure_mlp_ae_anomaly_classifier_float32.tflite"
METADATA_PATH = MODEL_DIR / "metadata.json"
THRESHOLDS_PATH = MODEL_DIR / "thresholds.json"


def select_file(title, filetypes, initialdir=None):
    root = tk.Tk()
    root.withdraw()
    root.attributes("-topmost", True)

    path = filedialog.askopenfilename(
        title=title,
        initialdir=str(initialdir or Path.cwd()),
        filetypes=filetypes
    )

    root.destroy()
    return Path(path) if path else None


if not MODEL_PATH.is_file():
    print("Saved DANN INT8 model not found in default directory.")

    MODEL_PATH = select_file(
        "Select trained DANN INT8 TFLite model",
        [("TFLite model", "*.tflite")]
    )

    if MODEL_PATH is None:
        raise RuntimeError("No DANN model selected.")

    MODEL_DIR = MODEL_PATH.parent
    OUTPUT_ROOT = MODEL_DIR.parent
    METADATA_PATH = MODEL_DIR / "metadata.json"
    THRESHOLDS_PATH = MODEL_DIR / "thresholds.json"


if not METADATA_PATH.is_file():
    raise FileNotFoundError(METADATA_PATH)

with open(METADATA_PATH, encoding="utf-8") as f:
    meta = json.load(f)

if THRESHOLDS_PATH.is_file():
    with open(THRESHOLDS_PATH, encoding="utf-8") as f:
        thresholds = json.load(f)
else:
    thresholds = meta["thresholds"]

if thresholds.get("calibration_source") == "int8_tflite":
    print(
        "WARNING: Reconstruction thresholds were calibrated for INT8. "
        "Float32 reconstruction predictions may need separate calibration."
    )


# ==========================================================
# 2. LOAD SAVED CONFIGURATION
# ==========================================================

FEATURES = meta["feature_columns"]
LABELS = meta["labels"]
SAFE_LABELS = meta["safe_labels"]
FAULT_LABELS = meta["fault_labels"]

WIN, NF = map(int, meta["input_shape"])
STEP = int(meta["window_step"])
ROW_SEC = float(meta["row_period_sec"])
SCALE = float(meta["scale_divisor"])

MEAN = np.asarray(
    meta["normalization_mean"], dtype=np.float32
)

VAR = np.asarray(
    meta["normalization_variance"], dtype=np.float32
)

ANOM_THR = float(
    thresholds["anomaly_probability_threshold"]
)

CLASS_THR = float(
    thresholds["class_confidence_threshold"]
)

GLOBAL_RECON_THR = float(
    thresholds["global_reconstruction_fallback"]
)

GLOBAL_BASELINE = float(
    thresholds["global_normal_baseline"]
)

STATE_THRESHOLDS = thresholds[
    "state_reconstruction_thresholds"
]

PWM_EDGES = np.asarray(
    thresholds["pwm_bin_edges"], dtype=np.float32
)

PWM_NAMES = thresholds["pwm_bin_names"]

NORMAL_PWM_THRESHOLDS = thresholds[
    "normal_pwm_reconstruction_thresholds"
]

LOCAL_BASELINES = thresholds["local_normal_baselines"]

# Original DANN Cell 2 severity settings
MILD = 2.0
MODERATE = 4.0
SEVERE = 8.0

BATCH_SIZE = 128

print("DANN model:", MODEL_PATH)
print("Model input:", WIN, "x", NF)
print("Anomaly threshold:", ANOM_THR)
print("Class confidence threshold:", CLASS_THR)


# ==========================================================
# 3. SELECT MANUAL TEST FILE
# ==========================================================

TEST_FILE = select_file(
    "Select motor CSV or Excel file for DANN manual testing",
    [
        ("Motor data files", "*.xlsx *.csv"),
        ("Excel files", "*.xlsx"),
        ("CSV files", "*.csv"),
        ("All files", "*.*")
    ]
)

if TEST_FILE is None:
    raise RuntimeError("Manual test cancelled.")

RESULT_DIR = OUTPUT_ROOT / "manual_test_results"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

stem = TEST_FILE.stem

print("\nSelected file:", TEST_FILE)
print("Results directory:", RESULT_DIR)


# ==========================================================
# 4. READ AND PREPROCESS DATA — SAME AS DANN CELL 4
# ==========================================================

if TEST_FILE.suffix.lower() == ".xlsx":
    df = pd.read_excel(TEST_FILE)

elif TEST_FILE.suffix.lower() == ".csv":
    df = pd.read_csv(TEST_FILE)

else:
    raise ValueError("Only CSV and XLSX files supported.")

missing = [c for c in FEATURES if c not in df.columns]

if missing:
    raise ValueError(f"Missing required feature columns: {missing}")

data = df[FEATURES].apply(
    pd.to_numeric, errors="coerce"
)

data = data / SCALE

# Same PWM clipping as original DANN preprocessing.
data["PWM"] = data["PWM"].clip(upper=1.0)

data = data.replace(
    [np.inf, -np.inf], np.nan
).dropna().reset_index(drop=True)

X = data.to_numpy(dtype=np.float32)

if len(X) < WIN:
    raise ValueError(
        f"Only {len(X)} valid rows. Minimum {WIN} required."
    )

starts = np.arange(
    0, len(X) - WIN + 1, STEP, dtype=np.int64
)

windows = np.stack([
    X[s:s + WIN] for s in starts
]).astype(np.float32)

print("Valid rows:", len(X))
print("Total windows:", len(windows))


# ==========================================================
# 5. LOAD DANN FLOAT32 TFLITE MODEL
# ==========================================================

interpreter = tf.lite.Interpreter(model_path=str(MODEL_PATH))
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()

if len(input_details) != 1:
    raise RuntimeError("Expected exactly one model input.")

input_detail = input_details[0]

if input_detail["dtype"] != np.float32:
    raise TypeError(
        "Selected model is not Float32. "
        "Choose pure_mlp_ae_anomaly_classifier_float32.tflite."
    )


def output_role(detail):
    shape = tuple(int(v) for v in detail["shape"])
    elements = int(np.prod(shape[1:]))

    if elements == WIN * NF:
        return "reconstruction_output"

    if elements == len(LABELS):
        return "class_output"

    if elements == 1:
        return "anomaly_output"

    raise RuntimeError(
        f"Unknown model output: {detail['name']} {shape}"
    )


# ==========================================================
# 6. RUN DANN FLOAT32 INFERENCE
# ==========================================================

outputs = {
    "reconstruction_output": [],
    "anomaly_output": [],
    "class_output": []
}

for start in range(0, len(windows), BATCH_SIZE):

    batch = windows[start:start + BATCH_SIZE].astype(np.float32)

    desired_shape = [len(batch), WIN, NF]

    if list(input_detail["shape"]) != desired_shape:
        interpreter.resize_tensor_input(
            input_detail["index"],
            desired_shape,
            strict=False
        )
        interpreter.allocate_tensors()
        input_detail = interpreter.get_input_details()[0]

    interpreter.set_tensor(input_detail["index"], batch)
    interpreter.invoke()

    for detail in interpreter.get_output_details():

        role = output_role(detail)

        value = interpreter.get_tensor(
            detail["index"]
        ).astype(np.float32)

        if role == "reconstruction_output":
            value = value.reshape(-1, WIN, NF)
        elif role == "class_output":
            value = value.reshape(-1, len(LABELS))
        else:
            value = value.reshape(-1)

        outputs[role].append(value)

predictions = {
    role: np.concatenate(values, axis=0)
    for role, values in outputs.items()
}

reconstructed = predictions["reconstruction_output"]
class_probability = predictions["class_output"]

# DANN anomaly head produces RAW LOGITS.
anomaly_logit = predictions["anomaly_output"]

# Convert logits to probabilities exactly once.
anomaly_probability = np.exp(
    -np.logaddexp(
        0.0,
        -anomaly_logit.astype(np.float64)
    )
).astype(np.float32)

print("Float32 DANN inference completed.")
print("Windows processed:", len(windows))


# ==========================================================
# 7. CALCULATE RECONSTRUCTION MSE
# ==========================================================

normalized = (
    windows - MEAN.reshape(1, 1, NF)
) / np.sqrt(
    VAR.reshape(1, 1, NF) + 1e-7
)

reconstruction_mse = np.mean(
    np.square(reconstructed - normalized),
    axis=(1, 2)
).astype(np.float32)

PWM_INDEX = FEATURES.index("PWM")
IRMS_INDEX = FEATURES.index(
    thresholds.get("load_proxy_feature", "Irms")
)

mean_pwm = windows[:, :, PWM_INDEX].mean(axis=1)
mean_irms = windows[:, :, IRMS_INDEX].mean(axis=1)

class_id = np.argmax(
    class_probability, axis=1
).astype(np.int64)

class_confidence = np.max(
    class_probability, axis=1
)


# ==========================================================
# 8. DANN STATE/PWM RECONSTRUCTION THRESHOLDS
# ==========================================================

def pwm_bin_index(pwm):
    return int(np.clip(
        np.searchsorted(
            PWM_EDGES, pwm, side="right"
        ) - 1,
        0,
        len(PWM_NAMES) - 1
    ))


def reconstruction_threshold(cid, pwm, irms):
    label = LABELS[int(cid)]

    if label in ["motor_off", "turning_off", "turning_on"]:
        return float(
            STATE_THRESHOLDS.get(label, GLOBAL_RECON_THR)
        )

    name = PWM_NAMES[pwm_bin_index(pwm)]

    return float(
        NORMAL_PWM_THRESHOLDS.get(
            name, GLOBAL_RECON_THR
        )
    )


recon_thresholds = np.asarray([
    reconstruction_threshold(cid, pwm, irms)
    for cid, pwm, irms in zip(
        class_id, mean_pwm, mean_irms
    )
], dtype=np.float32)


# ==========================================================
# 9. ORIGINAL DANN THREE-HEAD DECISION FUSION
# ==========================================================

binary_flags = anomaly_probability >= ANOM_THR
reconstruction_flags = reconstruction_mse > recon_thresholds

known_fault_flags = (
    np.isin(
        [LABELS[int(i)] for i in class_id],
        FAULT_LABELS
    )
    & (class_confidence >= CLASS_THR)
)

anomaly_flags = (
    binary_flags
    | reconstruction_flags
    | known_fault_flags
)

final_decisions = []

for i in range(len(class_id)):

    label = LABELS[int(class_id[i])]

    if known_fault_flags[i]:
        decision = label

    elif binary_flags[i] or reconstruction_flags[i]:
        decision = "unknown_fault"

    elif (
        label in SAFE_LABELS
        and class_confidence[i] >= CLASS_THR
    ):
        decision = label

    else:
        decision = "uncertain_safe"

    final_decisions.append(decision)


# ==========================================================
# 10. DANN RECONSTRUCTION SEVERITY
# ==========================================================

local_baselines = np.asarray([
    float(
        LOCAL_BASELINES.get(
            PWM_NAMES[pwm_bin_index(pwm)],
            GLOBAL_BASELINE
        )
    )
    for pwm in mean_pwm
], dtype=np.float32)

safe_baselines = np.where(
    local_baselines > 0,
    local_baselines,
    1.0
)

severity_ratio = reconstruction_mse / safe_baselines

severity_labels = []
recommended_actions = []

for ratio in severity_ratio:

    if ratio < MILD:
        severity_labels.append("normal")
        recommended_actions.append("No action needed.")

    elif ratio < MODERATE:
        severity_labels.append("mild")
        recommended_actions.append(
            "Log and monitor; no immediate action."
        )

    elif ratio < SEVERE:
        severity_labels.append("moderate")
        recommended_actions.append(
            "Schedule inspection soon."
        )

    else:
        severity_labels.append("severe")
        recommended_actions.append(
            "Stop motor / inspect immediately."
        )


# ==========================================================
# 11. EARLY WARNING — ADDITIONAL OUTPUT
# ==========================================================

early_candidates = (
    (class_id == LABELS.index("normal"))
    & reconstruction_flags
)

early_streak = np.zeros(
    len(early_candidates), dtype=np.int32
)

for i, candidate in enumerate(early_candidates):
    if candidate:
        early_streak[i] = (
            early_streak[i - 1] + 1 if i > 0 else 1
        )

early_flags = early_streak >= 3


# ==========================================================
# 12. CREATE DETAILED RESULT CSV
# ==========================================================

time_sec = (starts + WIN / 2.0) * ROW_SEC

result = pd.DataFrame({

    "window": np.arange(len(windows)),
    "start_row": starts,
    "time_sec": time_sec,

    "mean_pwm": mean_pwm,
    "mean_irms": mean_irms,

    "pred_class_id": class_id,
    "pred_label": [
        LABELS[int(i)] for i in class_id
    ],

    "class_confidence": class_confidence,

    "anomaly_logit": anomaly_logit,
    "anomaly_probability": anomaly_probability,

    "binary_anomaly_flag": binary_flags.astype(int),

    "reconstruction_mse": reconstruction_mse,
    "reconstruction_threshold": recon_thresholds,

    "local_baseline": local_baselines,
    "severity_ratio": severity_ratio,
    "severity_label": severity_labels,
    "recommended_action": recommended_actions,

    "reconstruction_anomaly_flag": reconstruction_flags.astype(int),
    "anomaly_flag": anomaly_flags.astype(int),
    "final_decision": final_decisions,

    "early_warning_candidate": early_candidates.astype(int),
    "early_warning_streak": early_streak,
    "early_warning_flag": early_flags.astype(int)
})

for i, label in enumerate(LABELS):
    result[f"probability_{label}"] = (
        class_probability[:, i]
    )

CSV_PATH = RESULT_DIR / f"manual_test_{stem}.csv"

result.to_csv(CSV_PATH, index=False)


# ==========================================================
# 13. DISPLAY DANN TEST SUMMARY
# ==========================================================

print("\n" + "=" * 70)
print("MANUAL DANN / PURE MLP ANN 3-HEAD TEST — FLOAT32")
print("=" * 70)

print("File:", TEST_FILE.name)
print("Total windows:", len(result))

print(
    "Majority class:",
    result["pred_label"].value_counts().idxmax()
)

print(
    "Majority final decision:",
    result["final_decision"].value_counts().idxmax()
)

print(
    "Majority severity:",
    result["severity_label"].value_counts().idxmax()
)

print(
    "Mean class confidence:",
    float(np.mean(class_confidence))
)

print(
    "Mean anomaly probability:",
    float(np.mean(anomaly_probability))
)

print(
    "Mean reconstruction MSE:",
    float(np.mean(reconstruction_mse))
)

print(
    "Anomalous windows:",
    int(anomaly_flags.sum()), "/", len(result)
)

print(
    "Early-warning windows:",
    int(early_flags.sum()), "/", len(result)
)

if early_flags.any():
    first_time = float(time_sec[np.flatnonzero(early_flags)[0]])
    print("First EARLY_WARNING:", first_time, "s")
else:
    print("No EARLY_WARNING generated.")

print("\nFinal decisions:")
print(result["final_decision"].value_counts())

print("\nSeverity tiers:")
print(result["severity_label"].value_counts())

print("\nSaved CSV:", CSV_PATH)


# ==========================================================
# 14. SAVE FIVE MANUAL-TEST CHARTS
# ==========================================================

def save_chart(suffix):
    path = RESULT_DIR / f"manual_test_{stem}_{suffix}.png"
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.show()
    plt.close()
    print("Saved:", path)


# CHART 1 — DANN classification probabilities

plt.figure(figsize=(14, 7))

for i, label in enumerate(LABELS):
    plt.plot(
        time_sec,
        class_probability[:, i],
        label=label,
        linewidth=1.5
    )

plt.axhline(
    CLASS_THR,
    linestyle="--",
    label=f"Confidence threshold ({CLASS_THR:.2f})"
)

plt.xlabel("Time [s]")
plt.ylabel("Class probability")
plt.ylim(0, 1.05)
plt.title(f"DANN classification head probabilities — {stem}")
plt.legend(loc="upper right", ncol=2)
plt.grid(True, alpha=0.3)

save_chart("class_head")


# CHART 2 — DANN binary anomaly probability

plt.figure(figsize=(14, 5))

plt.plot(
    time_sec,
    anomaly_probability,
    label="Anomaly probability",
    linewidth=2
)

plt.axhline(
    ANOM_THR,
    linestyle="--",
    label=f"Anomaly threshold ({ANOM_THR:.3f})"
)

plt.fill_between(
    time_sec,
    0,
    1,
    where=binary_flags,
    alpha=0.15,
    label="Anomaly detected"
)

plt.xlabel("Time [s]")
plt.ylabel("Anomaly probability")
plt.ylim(0, 1.05)
plt.title(f"DANN binary anomaly head — {stem}")
plt.legend()
plt.grid(True, alpha=0.3)

save_chart("anomaly_head")


# CHART 3 — DANN reconstruction MSE

plt.figure(figsize=(14, 5))

plt.plot(
    time_sec,
    reconstruction_mse,
    label="Reconstruction MSE",
    linewidth=2
)

plt.plot(
    time_sec,
    recon_thresholds,
    linestyle="--",
    linewidth=2,
    label="State/PWM-conditioned threshold"
)

plt.fill_between(
    time_sec,
    0,
    reconstruction_mse,
    where=reconstruction_flags,
    alpha=0.15,
    label="Reconstruction anomaly"
)

plt.xlabel("Time [s]")
plt.ylabel("Reconstruction MSE")
plt.title(f"DANN autoencoder reconstruction — {stem}")
plt.legend()
plt.grid(True, alpha=0.3)

save_chart("reconstruction_head")


# CHART 4 — DANN predicted class timeline

plt.figure(figsize=(14, 5))

plt.step(
    time_sec,
    class_id,
    where="mid",
    linewidth=2
)

plt.yticks(np.arange(len(LABELS)), LABELS)
plt.xlabel("Time [s]")
plt.ylabel("Predicted class")
plt.title(f"DANN classification result timeline — {stem}")
plt.grid(True, alpha=0.3)

save_chart("class_timeline")


# CHART 5 — DANN reconstruction severity

plt.figure(figsize=(14, 5))

plt.plot(
    time_sec,
    severity_ratio,
    linewidth=2,
    color="darkorange",
    label="Severity ratio (MSE / local baseline)"
)

for level, color in [
    (MILD, "gold"),
    (MODERATE, "orange"),
    (SEVERE, "red")
]:
    plt.axhline(
        level,
        linestyle="--",
        color=color,
        alpha=0.7
    )

plt.xlabel("Time [s]")
plt.ylabel("Severity ratio")
plt.title(f"DANN reconstruction severity — {stem}")
plt.legend()
plt.grid(True, alpha=0.3)

save_chart("severity")


# ==========================================================
# 15. COMPLETE
# ==========================================================

manual_result = result

print("\n" + "=" * 70)
print("DANN MANUAL TEST COMPLETED SUCCESSFULLY")
print("=" * 70)

print("Model:", MODEL_PATH.name)
print("Test file:", TEST_FILE.name)
print("CSV:", CSV_PATH)
print("Five charts saved in:", RESULT_DIR)
print("No training or recalibration performed.")